# ARRGO: Core Implementation

This notebook translates the mathematical and algorithmic specification of
ARRGO into an executable deterministic optimization framework.

The implementation follows the theoretical foundations established in the
previous notebooks and preserves the separation between:

- objective evaluation;
- regional representation;
- regional analysis;
- candidate generation;
- refinement action selection;
- global region selection;
- certified uncertainty and optimization potential;
- numerical robustness;
- budget management;
- and termination.

The first implementation targets one-dimensional bounded optimization.

The one-dimensional implementation provides a controlled environment in which
the correctness, consistency, and behavior of the core ARRGO mechanisms can be
tested before extending the framework to higher-dimensional domains.

The implementation does not introduce a new optimization principle that was
not defined in the previous design.

Instead, it translates the existing specification into executable components.

The implementation follows the principle:

$$
\boxed{
\text{Theory}
\rightarrow
\text{Data Structures}
\rightarrow
\text{Analysis}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Refinement}
\rightarrow
\text{Global Selection}
\rightarrow
\text{Termination}
}
$$

The objective function is treated as a deterministic black-box evaluator.

No analytical derivative, symbolic expression, or probabilistic model is
required by the core implementation.

Certified Mode additionally requires a valid Lipschitz constant and constructs
deterministic regional bounds from the available function evaluations.

The implementation must preserve the following fundamental properties:

- deterministic execution;
- persistent evaluation history;
- persistent region hierarchy;
- no-pruning;
- valid spatial refinement;
- explicit numerical tolerances;
- evaluation-budget accounting;
- monotonic incumbent improvement;
- and, in Certified Mode, validity of the certified global upper bound.

The implementation is developed incrementally so that each component can be
tested independently before being integrated into the complete ARRGO execution
loop.

## Implementation Architecture

The implementation of ARRGO is organized into independent components that
correspond directly to the conceptual responsibilities defined in the
algorithm design.

The purpose of this architecture is to prevent the optimization logic from
being concentrated inside one large procedure.

Each component should have a clearly defined responsibility and should expose
only the information required by the components that depend on it.

### Implementation Layers

The core implementation is organized conceptually into the following layers:

1. Objective Evaluation
2. Numerical Utilities
3. Evaluation History
4. Region Representation
5. Region Analysis
6. Candidate Generation
7. Refinement Action Selection
8. Global Region Selection
9. Certified Bounds
10. Termination
11. ARRGO Execution

The dependency direction should remain as simple as possible.

Lower-level components provide reusable numerical and data-management
operations, while higher-level components coordinate the optimization process.

### Objective Evaluation Layer

The objective function is represented as a deterministic callable.

Conceptually,

$$
f:\Omega\rightarrow\mathbb{R}.
$$

The optimizer does not require knowledge of the internal implementation of
\(f\).

The evaluation layer is responsible only for requesting objective values and
returning valid observations.

It must not decide:

- which region should be refined;
- which point should be sampled;
- which region is globally important;
- or when optimization should terminate.

These decisions belong to higher-level components.

### Numerical Utilities Layer

The numerical layer provides common operations required by the entire
implementation.

Its responsibilities include:

- point comparison;
- duplicate detection;
- boundary handling;
- interval validation;
- split-point validation;
- stable numerical comparisons;
- tolerance handling;
- and finite-value validation.

The numerical policy must be centralized rather than independently
reimplemented by different components.

This ensures that the same numerical rules are applied consistently throughout
ARRGO.

### Evaluation History Layer

The evaluation history stores all objective evaluations performed by ARRGO.

Conceptually,

$$
D_t
=
\left\{
(x_i,f(x_i))
\right\}_{i=1}^{N_t}.
$$

The history is persistent.

Therefore,

$$
D_t
\subseteq
D_{t+1}.
$$

The history layer is responsible for:

- storing observations;
- preventing duplicate evaluations;
- retrieving observations relevant to a region;
- maintaining deterministic ordering;
- and reporting the total number of evaluations.

The history layer does not determine optimization priority.

### Region Representation Layer

A region represents a spatial unit of the search domain.

For the one-dimensional implementation,

$$
R=[l,r].
$$

A region maintains its structural identity and its relationship with the region
hierarchy.

Conceptually, a region contains:

$$
R
=
(
\mathrm{Id},
l,
r,
\mathrm{ParentId},
\mathrm{ChildrenIds},
D_R,
B_R,
Q_R,
U_R,
P_R,
\mathrm{State}
).
$$

Not every field must be stored as a permanent numerical value.

Some fields may be derived or recomputed from the current global information.

The important requirement is that the region retains its structural identity
and historical relationships.

### Region Hierarchy Layer

The hierarchy stores all regions created during the optimization process.

The root region is

$$
R_0=\Omega.
$$

When a region is split,

$$
R_p
\rightarrow
\{R_L,R_R\},
$$

the parent remains in the hierarchy.

Therefore,

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

This is the implementation-level representation of the no-pruning
requirement.

A region may become REFINED or STABLE without being deleted.

### Region Analysis Layer

Region analysis derives the current information state of a region from the
available observations and global context.

Its responsibilities include:

- ordering regional samples;
- computing spatial gaps;
- computing observed secant slopes;
- detecting directional changes;
- estimating observed slope variation;
- identifying candidate extrema;
- evaluating information sufficiency;
- and computing certified quantities when Certified Mode is active.

The analysis layer must not directly execute sampling or splitting.

Its purpose is to answer:

> What is currently known about this region, and what information remains
> unresolved?

### Candidate Generation Layer

Candidate generation transforms unresolved information into possible
refinement locations.

The sampling candidate set is

$$
C_{\mathrm{sample}}(R).
$$

The splitting candidate set is

$$
C_{\mathrm{split}}(R).
$$

Candidates may originate from:

- spatial coverage gaps;
- observed behavioral transitions;
- observed slope variation;
- certified uncertainty;
- certified potential;
- and valid boundaries.

Every candidate retains sufficient provenance to identify why it was
generated.

Candidate generation does not evaluate unknown objective values.

### Candidate Evaluation Layer

Candidate evaluation determines the information profile of each candidate
without requiring the unknown objective value at that candidate.

For a sampling candidate \(x_c\), the conceptual profile is

$$
I(x_c)
=
\left(
I_{\mathrm{coverage}},
I_{\mathrm{behavior}},
I_{\mathrm{uncertainty}},
I_{\mathrm{potential}}
\right).
$$

For a split candidate \(s_c\), the structural profile is

$$
V_{\mathrm{split}}(s_c)
=
\left(
V_{\mathrm{coverage}},
V_{\mathrm{behavior}},
V_{\mathrm{uncertainty}},
V_{\mathrm{potential}}
\right).
$$

These profiles support multi-criteria comparison without requiring an
arbitrary weighted scalar objective.

### Refinement Action Layer

The action-selection component determines whether an eligible region should
be:

$$
\mathrm{Sample},
\qquad
\mathrm{Split},
\qquad
\mathrm{Stable}.
$$

The decision depends on:

- unresolved information;
- available candidates;
- evaluation budget;
- active objectives;
- and current global relevance.

The action layer does not independently select the globally most important
region.

It operates on a region that has already been considered for refinement.

### Global Selection Layer

Global selection determines which eligible region receives the next refinement
opportunity.

The global selection component uses the current global state together with
regional information.

Conceptually,

$$
R_t^*
=
\operatorname{SelectRegion}
\left(
\mathcal{R}_t,G_t
\right).
$$

The selected region must be globally relevant and locally actionable.

In Certified Mode, regional optimization potential provides an important
source of global relevance.

The global selection layer must preserve fairness so that persistently
relevant regions cannot be permanently ignored.

### Certified Bounds Layer

Certified bounds are implemented separately from empirical behavioral
analysis.

When a valid Lipschitz constant \(L\) is available, the regional upper
envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

The regional potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The certified gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

This layer must preserve the validity of the mathematical enclosure.

An estimated Lipschitz constant must never automatically be treated as a
certified constant.

### Termination Layer

Termination evaluates whether ARRGO should stop.

The termination layer considers:

- Certified tolerance;
- evaluation-budget exhaustion;
- numerical limitations;
- and other explicitly configured stopping conditions.

Certified termination is based on

$$
\Delta_{\mathrm{global}}
\le
\epsilon.
$$

Budget termination is based on

$$
N_t
\ge
N_{\max}.
$$

These conditions must remain logically distinct.

### ARRGO Execution Layer

The execution layer coordinates all lower-level components.

Its responsibility is to implement the complete iteration:

$$
\boxed{
\begin{aligned}
&\text{Analyze}\\
&\rightarrow
\text{Identify Objectives}\\
&\rightarrow
\text{Generate Candidates}\\
&\rightarrow
\text{Determine Feasible Actions}\\
&\rightarrow
\text{Select Global Region}\\
&\rightarrow
\text{Select Candidate}\\
&\rightarrow
\text{Execute Refinement}\\
&\rightarrow
\text{Update State}\\
&\rightarrow
\text{Check Termination}.
\end{aligned}
}
$$

The execution layer coordinates these operations but should not duplicate the
mathematical logic implemented by the individual components.

### Separation of Responsibilities

The implementation must maintain the following separation:

| Component | Primary Responsibility |
| --- | --- |
| Objective Evaluator | Obtain objective values |
| Numerical Utilities | Maintain numerical consistency |
| Evaluation History | Persist all evaluations |
| Region | Represent one spatial region |
| Region Hierarchy | Preserve structural history |
| Region Analyzer | Derive regional information |
| Candidate Generator | Produce valid refinement candidates |
| Candidate Evaluator | Quantify candidate information profiles |
| Action Selector | Choose Sample, Split, or Stable |
| Global Selector | Choose the next globally relevant region |
| Certified Bounds | Construct valid deterministic bounds |
| Termination Manager | Determine whether execution should stop |
| ARRGO Engine | Coordinate the complete optimization cycle |

No component should silently assume responsibilities belonging to another
component.

### Data Flow

The primary information flow is

$$
\text{Objective Evaluator}
\rightarrow
\text{Evaluation History}
\rightarrow
\text{Region Analysis}
\rightarrow
\text{Candidate Generation}
\rightarrow
\text{Candidate Evaluation}
\rightarrow
\text{Action Selection}.
$$

The global context then connects local decisions:

$$
\text{Regional States}
\rightarrow
\text{Global Selection}
\rightarrow
\text{Selected Region}
\rightarrow
\text{Selected Action}.
$$

After execution, information flows back into the persistent state:

$$
\text{Refinement}
\rightarrow
\text{Updated History}
\rightarrow
\text{Updated Regions}
\rightarrow
\text{Updated Global State}.
$$

This creates the adaptive feedback loop that drives ARRGO.

### Empirical and Certified Components

Not every implementation component is mode-specific.

The following components are common to both modes:

- objective evaluation;
- evaluation history;
- region representation;
- region hierarchy;
- behavioral analysis;
- candidate generation;
- action selection;
- global selection;
- budget management;
- numerical robustness;
- and termination management.

Certified Mode additionally activates:

- Lipschitz-based envelopes;
- regional uncertainty;
- regional optimization potential;
- global potential;
- and certified optimality-gap computation.

Therefore, Certified Mode extends the core framework rather than defining a
separate optimizer.

### Configuration Separation

Algorithm configuration must be separated from mutable optimization state.

Configuration contains fixed parameters such as:

- search-domain boundaries;
- evaluation budget;
- refinement factor;
- numerical tolerances;
- execution mode;
- and optional Lipschitz information.

Optimization state contains changing information such as:

- evaluated points;
- objective values;
- regions;
- incumbent;
- region states;
- certified bounds;
- and iteration counters.

Conceptually,

$$
\mathrm{ARRGOState}_t
\neq
\mathrm{ARRGOConfig}.
$$

Keeping these concepts separate makes the implementation easier to test and
reduces accidental modification of fixed algorithm parameters.

### Deterministic Execution

Every component must follow deterministic rules.

Given identical:

$$
G_0
\quad\text{and}\quad
\Theta,
$$

the same sequence of valid decisions should be produced, subject to explicitly
defined floating-point behavior.

No optimization decision should depend on:

- random sampling;
- unspecified collection ordering;
- uncontrolled external state;
- or implicit nondeterministic tie-breaking.

### Testing Strategy

The architecture also supports component-level validation.

Each major component can be tested independently before integration.

Examples include:

- region-boundary validation;
- duplicate detection;
- secant-slope calculation;
- candidate generation;
- contraction validation;
- envelope construction;
- exact potential computation;
- action selection;
- global selection;
- and termination logic.

Integration tests can then verify that these components preserve the global
ARRGO invariants.

### Implementation Principle

The implementation architecture follows the principle

$$
\boxed{
\text{Separate Information Acquisition, Spatial Representation, Analysis,
Decision Making, and Execution While Preserving a Single Persistent Global
Optimization State.}
}
$$

This architecture provides the structural foundation for implementing the
individual ARRGO components.

The next section defines the concrete core data structures that will be used
to represent evaluations, regions, configurations, and global optimization
state.

## Core Data Structures

The ARRGO implementation requires a set of explicit data structures that
represent the persistent optimization state and the information associated
with each region.

The data structures must reflect the mathematical objects defined in the
previous notebooks without introducing unnecessary duplication.

The core structures are:

- objective evaluation;
- evaluation history;
- region;
- region hierarchy;
- regional analysis state;
- ARRGO configuration;
- global optimization state;
- and certified state.

### Objective Evaluation

A single objective evaluation represents one evaluated point and its observed
objective value.

Conceptually,

$$
e_i
=
(x_i,y_i),
$$

where

$$
y_i=f(x_i).
$$

The point satisfies

$$
x_i\in\Omega.
$$

Under the current theoretical model, \(y_i\) must be a finite real value.

An evaluation should also retain enough information to identify its position
in the persistent evaluation history.

The evaluation object therefore conceptually contains:

$$
e_i
=
(
x_i,
y_i,
\mathrm{Id}
).
$$

The identifier is not an optimization criterion.

It provides deterministic traceability inside the implementation.

### Evaluation History

The evaluation history contains every valid objective evaluation performed by
ARRGO.

At iteration \(t\),

$$
D_t
=
\{e_1,e_2,\ldots,e_{N_t}\}.
$$

The history is persistent:

$$
D_t
\subseteq
D_{t+1}.
$$

The evaluation history is the authoritative source for the number of
objective evaluations.

Therefore,

$$
N_t=|D_t|.
$$

The history must support:

- insertion of a new valid evaluation;
- duplicate detection;
- deterministic ordering;
- retrieval of evaluations relevant to a region;
- and evaluation-count reporting.

The history must not remove an evaluation because a region is later split or
becomes inactive.

### Region

A region represents a spatial subset of the search domain.

For the initial one-dimensional implementation,

$$
R=[l,r].
$$

A region requires a persistent identity.

Conceptually,

$$
R
=
(
\mathrm{Id},
l,
r,
\mathrm{ParentId},
\mathrm{ChildrenIds},
\mathrm{State}
).
$$

The spatial boundaries satisfy

$$
a\le l<r\le b.
$$

The parent identifier is null for the root region.

Child identifiers reference regions created by structural refinement.

### Region Evaluation View

A region does not own an independent copy of the global evaluation history.

Instead, its local evaluation set is derived from the persistent global history.

Conceptually,

$$
D_R
=
\{(x_i,y_i)\in D_t:x_i\in R\}.
$$

This avoids inconsistent duplicated copies of the same objective evaluation.

The region therefore acts as a spatial view over the global evaluation history.

### Region Behavior Information

Observed behavior is represented separately from raw evaluations.

Conceptually,

$$
B_R
=
(
S_R,
\Delta S_R,
D_R^{\mathrm{direction}},
E_R^{\mathrm{extrema}},
C_R^{\mathrm{complexity}}
),
$$

where:

- \(S_R\) represents observed secant slopes;
- \(\Delta S_R\) represents observed slope variations;
- \(D_R^{\mathrm{direction}}\) represents directional states;
- \(E_R^{\mathrm{extrema}}\) represents candidate extrema;
- \(C_R^{\mathrm{complexity}}\) represents observed behavioral complexity.

These values are derived information.

They may therefore be recomputed whenever the regional evaluation view changes.

### Unresolved Information Profile

Each region maintains an unresolved information profile

$$
Q_R
=
(
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
).
$$

The components describe the current unresolved optimization information.

They are not required to be arbitrary numerical scores.

Instead, each component may contain the relevant measurements and resolution
state required by the corresponding decision logic.

The active objective set is

$$
O_R^*
\subseteq
\{
\mathrm{Coverage},
\mathrm{Behavior},
\mathrm{Uncertainty},
\mathrm{Potential}
\}.
$$

This allows the implementation to preserve multi-objective information
without introducing an arbitrary weighted scalar.

### Region State

Each region has exactly one lifecycle state:

$$
\mathrm{State}_R
\in
\{
\mathrm{ACTIVE},
\mathrm{STABLE},
\mathrm{REFINED}
\}.
$$

The state describes the current structural or refinement status of the region.

It does not describe whether the region contains the global optimum.

In particular,

$$
\mathrm{STABLE}
\neq
\text{Globally Optimal}.
$$

A stable region remains in the hierarchy and may become active again if its
global relevance changes.

### Region Hierarchy

The region hierarchy maintains all regions created during execution.

Conceptually,

$$
\mathcal{R}_t
=
\{R_0,R_1,\ldots,R_m\}.
$$

The root is

$$
R_0=\Omega.
$$

If a region is split,

$$
R_p
\rightarrow
\{R_L,R_R\},
$$

the parent remains in \(\mathcal{R}_t\).

Therefore,

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1}.
$$

The hierarchy must support:

- region lookup by identifier;
- parent-child relationships;
- containment;
- deterministic traversal;
- and retrieval of eligible regions.

### ARRGO Configuration

The configuration contains fixed parameters controlling one execution.

Conceptually,

$$
\Theta
=
(
\Omega,
N_{\max},
\rho,
\tau_x,
\tau_b,
\tau_f,
\mathrm{Mode},
L,
\epsilon
).
$$

Here:

- \(\Omega\) is the search domain;
- \(N_{\max}\) is the evaluation budget;
- \(\rho\) is the contraction parameter;
- \(\tau_x\) is the point/duplicate tolerance;
- \(\tau_b\) is the boundary tolerance;
- \(\tau_f\) is the numerical objective comparison tolerance;
- \(\mathrm{Mode}\) specifies Empirical or Certified execution;
- \(L\) is the optional valid Lipschitz constant;
- \(\epsilon\) is the requested optimization tolerance.

The configuration must remain immutable during an execution.

### Configuration Validity

The implementation must validate configuration before execution.

At minimum,

$$
a<b,
$$

$$
N_{\max}\ge0,
$$

and

$$
0<\rho<1.
$$

Numerical tolerances must be non-negative.

Certified Mode additionally requires a valid positive Lipschitz bound:

$$
L>0.
$$

If Certified Mode is selected without a valid Lipschitz bound, the execution
must be rejected rather than silently falling back to an empirical
interpretation.

### Global Optimization State

The global state contains the mutable information required by the execution
loop.

Conceptually,

$$
G_t
=
(
\mathcal{R}_t,
D_t,
x_{\mathrm{best}}^{(t)},
f_{\mathrm{best}}^{(t)},
N_t,
t
).
$$

The state therefore contains:

- the persistent region hierarchy;
- the persistent evaluation history;
- the incumbent;
- the best observed objective value;
- the evaluation count;
- and the iteration counter.

The state changes after each refinement iteration.

### Incumbent Representation

Before the first valid evaluation, an incumbent does not exist.

After at least one evaluation,

$$
x_{\mathrm{best}}^{(t)}
\in
\operatorname*{arg\,max}_{x_i\in D_t}f(x_i).
$$

The corresponding value is

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in D_t}f(x_i).
$$

The implementation must therefore support an explicit uninitialized state
before the first objective evaluation.

This avoids assigning an artificial objective value such as zero or
negative infinity when the objective itself may legitimately take such
values.

### Certified Global State

Certified Mode extends the global state with valid regional potentials and
the global certified gap.

Conceptually,

$$
G_t^{\mathrm{cert}}
=
\left(
G_t,
\{P_R\}_{R\in\mathcal{R}_t},
P_{\mathrm{global}},
\Delta_{\mathrm{global}}
\right).
$$

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P_R.
$$

The certified gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

These values exist only when a valid certified bound is available.

### Regional Certified State

For a region \(R\), the certified state contains the lower and upper
envelopes.

The lower envelope is

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right],
$$

and the upper envelope is

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

The regional uncertainty is

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

The regional potential is

$$
P_R
=
\max_{x\in R}U_R(x).
$$

The implementation may store these quantities explicitly or compute them
from the current regional evaluations when required.

The storage strategy is an implementation decision, provided mathematical
consistency is preserved.

### Candidate Representation

Sampling and splitting candidates should be represented explicitly.

A sampling candidate can conceptually be represented as

$$
c_{\mathrm{sample}}
=
(
x_c,
S_c
),
$$

where \(x_c\) is the candidate location and \(S_c\) is its provenance.

A split candidate is similarly

$$
c_{\mathrm{split}}
=
(
s_c,
S_c
).
$$

The provenance records the information sources that generated the candidate.

For example,

$$
S_c
\subseteq
\{
\mathrm{Coverage},
\mathrm{Behavior},
\mathrm{Uncertainty},
\mathrm{Potential},
\mathrm{Boundary}
\}.
$$

Provenance supports analysis and debugging but must not be treated as
objective-function evidence.

### Candidate Profiles

Candidate evaluation produces an information profile.

For a sampling candidate,

$$
I(c)
=
(
I_{\mathrm{coverage}},
I_{\mathrm{behavior}},
I_{\mathrm{uncertainty}},
I_{\mathrm{potential}}
).
$$

For a split candidate,

$$
V(c)
=
(
V_{\mathrm{coverage}},
V_{\mathrm{behavior}},
V_{\mathrm{uncertainty}},
V_{\mathrm{potential}}
).
$$

These profiles are multi-dimensional.

The implementation must not automatically collapse them into a weighted sum
unless a mathematically justified scalarization is explicitly introduced.

### Action Representation

The refinement action is represented by

$$
A_R
\in
\{
\mathrm{Sample},
\mathrm{Split},
\mathrm{Stable}
\}.
$$

A selected sampling action references a valid sampling candidate.

A selected splitting action references a valid split candidate.

A Stable action contains no new candidate and does not perform an objective
evaluation.

### Termination Representation

Termination must be explicit.

Conceptually,

$$
\mathrm{TerminationReason}
\in
\{
\mathrm{CertifiedTolerance},
\mathrm{BudgetExhausted},
\mathrm{NumericalLimit},
\mathrm{NoValidAction},
\mathrm{ConfigurationError}
\}.
$$

Additional implementation-specific reasons may be introduced if their meaning
is clearly defined.

A termination reason must never imply a stronger guarantee than the condition
that actually caused termination.

### State versus Derived Information

The implementation should distinguish persistent state from derived
information.

Persistent information includes:

- objective evaluations;
- region identities;
- region hierarchy;
- configuration;
- and execution counters.

Derived information includes:

- secant slopes;
- behavioral profiles;
- unresolved objectives;
- candidate profiles;
- regional envelopes;
- and region priorities.

Derived information may be recomputed after updates.

This distinction reduces the risk of stale analytical information.

### Data Ownership

The evaluation history is the authoritative owner of objective observations.

The region hierarchy is the authoritative owner of structural relationships.

The configuration is the authoritative owner of fixed execution parameters.

The global state coordinates these objects.

Therefore, no component should silently create an independent conflicting
copy of authoritative information.

### Information Provenance

ARRGO distinguishes three information categories:

1. **Observed information** — directly obtained from objective evaluations.
2. **Derived information** — computed from observations and geometry.
3. **Certified information** — mathematically valid bounds derived under explicit
   assumptions.

These categories must not be confused.

For example, an observed slope reversal is derived evidence of possible local
behavior.

It is not a certified statement that an unobserved maximum exists.

Similarly, a regional potential is a certified upper bound only when its
underlying Lipschitz assumption is valid.

### Core Structural Invariants

The data structures must support the following invariants:

$$
D_t
\subseteq
D_{t+1},
$$

$$
\mathcal{R}_t
\subseteq
\mathcal{R}_{t+1},
$$

$$
N_t=|D_t|,
$$

$$
N_t\le N_{\max},
$$

and

$$
f_{\mathrm{best}}^{(t+1)}
\ge
f_{\mathrm{best}}^{(t)}.
$$

For every valid region,

$$
a\le l<r\le b.
$$

For every parent-child relationship,

$$
R_c\subseteq R_p.
$$

In Certified Mode,

$$
f_{\mathrm{best}}^{(t)}
\le
f^*
\le
P_{\mathrm{global}}^{(t)}.
$$

### Data Structure Design Principle

The core data structures follow the principle

$$
\boxed{
\text{Represent Persistent Observations and Structure Explicitly, Derive
Analytical Information From Them, and Keep Configuration, State, and
Certified Information Logically Separate.}
}
$$

These structures provide the foundation for the first executable ARRGO
components.

The next section implements the numerical configuration and tolerance
system that all subsequent components will use.

In [1]:
from dataclasses import dataclass
from enum import Enum


class ARRGOExecutionMode(str, Enum):
    EMPIRICAL = "empirical"
    CERTIFIED = "certified"


class TerminationReason(str, Enum):
    CERTIFIED_TOLERANCE = "certified_tolerance"
    BUDGET_EXHAUSTED = "budget_exhausted"
    NUMERICAL_LIMIT = "numerical_limit"
    NO_VALID_ACTION = "no_valid_action"
    CONFIGURATION_ERROR = "configuration_error"


@dataclass(frozen=True)
class NumericalTolerances:
    point: float = 1e-10
    boundary: float = 1e-10
    objective: float = 1e-10
    slope: float = 1e-10

    def __post_init__(self) -> None:
        values = {
            "point": self.point,
            "boundary": self.boundary,
            "objective": self.objective,
            "slope": self.slope,
        }

        for name, value in values.items():
            if value < 0:
                raise ValueError(
                    f"Numerical tolerance '{name}' must be non-negative."
                )


@dataclass(frozen=True)
class ARRGOConfig:
    lower_bound: float
    upper_bound: float
    max_evaluations: int
    contraction_factor: float = 0.5
    optimization_tolerance: float = 1e-6
    mode: ARRGOExecutionMode = ARRGOExecutionMode.EMPIRICAL
    lipschitz_constant: float | None = None
    tolerances: NumericalTolerances = NumericalTolerances()

    def __post_init__(self) -> None:
        if self.lower_bound >= self.upper_bound:
            raise ValueError(
                "The lower bound must be strictly smaller than the upper bound."
            )

        if self.max_evaluations < 0:
            raise ValueError(
                "The maximum number of evaluations must be non-negative."
            )

        if not 0 < self.contraction_factor < 1:
            raise ValueError(
                "The contraction factor must satisfy 0 < rho < 1."
            )

        if self.optimization_tolerance < 0:
            raise ValueError(
                "The optimization tolerance must be non-negative."
            )

        if self.mode == ARRGOExecutionMode.CERTIFIED:
            if self.lipschitz_constant is None:
                raise ValueError(
                    "Certified mode requires a valid Lipschitz constant."
                )

            if self.lipschitz_constant <= 0:
                raise ValueError(
                    "The Lipschitz constant must be positive in certified mode."
                )

    @property
    def domain(self) -> tuple[float, float]:
        return self.lower_bound, self.upper_bound

In [2]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Evaluation:
    id: int
    x: float
    value: float

    def __post_init__(self) -> None:
        if self.id < 0:
            raise ValueError(
                "Evaluation id must be non-negative."
            )

    @property
    def point(self) -> float:
        return self.x

    @property
    def objective_value(self) -> float:
        return self.value

In [3]:
from dataclasses import dataclass, field


@dataclass
class EvaluationHistory:
    point_tolerance: float
    _evaluations: list[Evaluation] = field(default_factory=list)

    def __post_init__(self) -> None:
        if self.point_tolerance < 0:
            raise ValueError(
                "Point tolerance must be non-negative."
            )

    @property
    def evaluations(self) -> tuple[Evaluation, ...]:
        return tuple(self._evaluations)

    @property
    def count(self) -> int:
        return len(self._evaluations)

    def contains_point(self, x: float) -> bool:
        return any(
            abs(evaluation.x - x) <= self.point_tolerance
            for evaluation in self._evaluations
        )

    def add(self, evaluation: Evaluation) -> None:
        if self.contains_point(evaluation.x):
            raise ValueError(
                f"An evaluation already exists near x={evaluation.x}."
            )

        self._evaluations.append(evaluation)

    def get_by_id(self, evaluation_id: int) -> Evaluation:
        for evaluation in self._evaluations:
            if evaluation.id == evaluation_id:
                return evaluation

        raise KeyError(
            f"Evaluation with id={evaluation_id} was not found."
        )

    def values_in_interval(
        self,
        lower_bound: float,
        upper_bound: float,
    ) -> tuple[Evaluation, ...]:
        if lower_bound > upper_bound:
            raise ValueError(
                "The lower bound must not exceed the upper bound."
            )

        return tuple(
            evaluation
            for evaluation in self._evaluations
            if lower_bound - self.point_tolerance
            <= evaluation.x
            <= upper_bound + self.point_tolerance
        )

In [4]:
from dataclasses import dataclass, field
from enum import Enum


class RegionState(str, Enum):
    ACTIVE = "active"
    STABLE = "stable"
    REFINED = "refined"


@dataclass
class Region:
    id: int
    lower_bound: float
    upper_bound: float
    parent_id: int | None = None
    children_ids: list[int] = field(default_factory=list)
    state: RegionState = RegionState.ACTIVE

    def __post_init__(self) -> None:
        if self.id < 0:
            raise ValueError(
                "Region id must be non-negative."
            )

        if self.lower_bound >= self.upper_bound:
            raise ValueError(
                "The lower bound must be strictly smaller "
                "than the upper bound."
            )

        if self.parent_id is not None and self.parent_id < 0:
            raise ValueError(
                "Parent region id must be non-negative."
            )

        if len(set(self.children_ids)) != len(self.children_ids):
            raise ValueError(
                "Children region ids must be unique."
            )

        if any(child_id < 0 for child_id in self.children_ids):
            raise ValueError(
                "Children region ids must be non-negative."
            )

    @property
    def interval(self) -> tuple[float, float]:
        return self.lower_bound, self.upper_bound

    @property
    def width(self) -> float:
        return self.upper_bound - self.lower_bound

    def contains(self, x: float, tolerance: float = 0.0) -> bool:
        if tolerance < 0:
            raise ValueError(
                "Tolerance must be non-negative."
            )

        return (
            self.lower_bound - tolerance
            <= x
            <= self.upper_bound + tolerance
        )

    def add_child(self, child_id: int) -> None:
        if child_id < 0:
            raise ValueError(
                "Child region id must be non-negative."
            )

        if child_id not in self.children_ids:
            self.children_ids.append(child_id)

    def mark_refined(self) -> None:
        self.state = RegionState.REFINED

    def mark_stable(self) -> None:
        self.state = RegionState.STABLE

    def activate(self) -> None:
        self.state = RegionState.ACTIVE

In [5]:
@dataclass
class RegionHierarchy:
    _regions: dict[int, Region] = field(default_factory=dict)

    @property
    def regions(self) -> tuple[Region, ...]:
        return tuple(self._regions.values())

    @property
    def count(self) -> int:
        return len(self._regions)

    def add(self, region: Region) -> None:
        if region.id in self._regions:
            raise ValueError(
                f"Region with id={region.id} already exists."
            )

        if region.parent_id is not None:
            if region.parent_id not in self._regions:
                raise KeyError(
                    f"Parent region with id={region.parent_id} "
                    "does not exist."
                )

        self._regions[region.id] = region

        if region.parent_id is not None:
            parent = self._regions[region.parent_id]
            parent.add_child(region.id)

    def get(self, region_id: int) -> Region:
        try:
            return self._regions[region_id]
        except KeyError as exc:
            raise KeyError(
                f"Region with id={region_id} was not found."
            ) from exc

    def has(self, region_id: int) -> bool:
        return region_id in self._regions

    def children_of(self, region_id: int) -> tuple[Region, ...]:
        region = self.get(region_id)

        return tuple(
            self._regions[child_id]
            for child_id in region.children_ids
        )

    def active_regions(self) -> tuple[Region, ...]:
        return tuple(
            region
            for region in self._regions.values()
            if region.state == RegionState.ACTIVE
        )

    def root_regions(self) -> tuple[Region, ...]:
        return tuple(
            region
            for region in self._regions.values()
            if region.parent_id is None
        )

In [6]:
from abc import ABC, abstractmethod
from typing import Callable


class ObjectiveFunction(ABC):
    @abstractmethod
    def evaluate(self, x: float) -> float:
        """Evaluate the deterministic objective function at x."""
        raise NotImplementedError


@dataclass(frozen=True)
class CallableObjective(ObjectiveFunction):
    function: Callable[[float], float]

    def evaluate(self, x: float) -> float:
        value = self.function(x)

        if not isinstance(value, (int, float)):
            raise TypeError(
                "Objective function must return a numeric value."
            )

        value = float(value)

        if not __import__("math").isfinite(value):
            raise ValueError(
                "Objective function must return a finite value."
            )

        return value

In [7]:
class ObjectiveEvaluator:
    def __init__(
        self,
        objective: ObjectiveFunction,
        history: EvaluationHistory,
        max_evaluations: int,
    ) -> None:
        if max_evaluations < 0:
            raise ValueError(
                "Maximum evaluations must be non-negative."
            )

        self._objective = objective
        self._history = history
        self._max_evaluations = max_evaluations

    @property
    def evaluation_count(self) -> int:
        return self._history.count

    @property
    def remaining_budget(self) -> int:
        return self._max_evaluations - self.evaluation_count

    @property
    def budget_exhausted(self) -> bool:
        return self.evaluation_count >= self._max_evaluations

    def can_evaluate(self, x: float) -> bool:
        if self.budget_exhausted:
            return False

        if self._history.contains_point(x):
            return False

        return True

    def evaluate(self, x: float) -> Evaluation:
        if self.budget_exhausted:
            raise RuntimeError(
                "The evaluation budget has been exhausted."
            )

        if self._history.contains_point(x):
            raise ValueError(
                f"Point x={x} has already been evaluated."
            )

        value = self._objective.evaluate(x)

        evaluation = Evaluation(
            id=self._history.count,
            x=x,
            value=value,
        )

        self._history.add(evaluation)

        return evaluation

In [8]:
@dataclass(frozen=True)
class RegionEvaluationView:
    region: Region
    evaluations: tuple[Evaluation, ...]

    @property
    def count(self) -> int:
        return len(self.evaluations)

    @property
    def is_empty(self) -> bool:
        return self.count == 0

    @property
    def points(self) -> tuple[float, ...]:
        return tuple(
            evaluation.x
            for evaluation in self.evaluations
        )

    @property
    def objective_values(self) -> tuple[float, ...]:
        return tuple(
            evaluation.value
            for evaluation in self.evaluations
        )

    @property
    def best_evaluation(self) -> Evaluation | None:
        if self.is_empty:
            return None

        return max(
            self.evaluations,
            key=lambda evaluation: evaluation.value,
        )


def build_region_evaluation_view(
    region: Region,
    history: EvaluationHistory,
    tolerance: float,
) -> RegionEvaluationView:
    evaluations = history.values_in_interval(
        region.lower_bound,
        region.upper_bound,
    )

    filtered = tuple(
        evaluation
        for evaluation in evaluations
        if region.contains(evaluation.x, tolerance)
    )

    return RegionEvaluationView(
        region=region,
        evaluations=filtered,
    )

In [9]:
@dataclass(frozen=True)
class RegionAnalysis:
    region_id: int
    width: float
    evaluation_count: int
    coverage_resolution: float
    value_range: float
    best_value: float | None
    best_point: float | None
    slope_range: float
    boundary_distance: float

    @property
    def has_evaluations(self) -> bool:
        return self.evaluation_count > 0

    @property
    def has_multiple_evaluations(self) -> bool:
        return self.evaluation_count >= 2


def analyze_region(
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> RegionAnalysis:
    region = view.region
    evaluations = view.evaluations

    if not evaluations:
        return RegionAnalysis(
            region_id=region.id,
            width=region.width,
            evaluation_count=0,
            coverage_resolution=region.width,
            value_range=0.0,
            best_value=None,
            best_point=None,
            slope_range=0.0,
            boundary_distance=region.width,
        )

    points = sorted(
        evaluation.x
        for evaluation in evaluations
    )

    values_by_point = sorted(
        evaluations,
        key=lambda evaluation: evaluation.x,
    )

    best_evaluation = max(
        evaluations,
        key=lambda evaluation: evaluation.value,
    )

    value_range = (
        max(evaluation.value for evaluation in evaluations)
        - min(evaluation.value for evaluation in evaluations)
    )

    if len(points) >= 2:
        gaps = [
            right - left
            for left, right in zip(points, points[1:])
        ]

        coverage_resolution = max(gaps)

        slopes = []

        for left, right in zip(
            values_by_point,
            values_by_point[1:],
        ):
            dx = right.x - left.x

            if dx > tolerances.point:
                slopes.append(
                    (right.value - left.value) / dx
                )

        if slopes:
            slope_range = max(slopes) - min(slopes)
        else:
            slope_range = 0.0
    else:
        coverage_resolution = region.width
        slope_range = 0.0

    boundary_distance = min(
        abs(points[0] - region.lower_bound),
        abs(region.upper_bound - points[-1]),
    )

    return RegionAnalysis(
        region_id=region.id,
        width=region.width,
        evaluation_count=len(evaluations),
        coverage_resolution=coverage_resolution,
        value_range=value_range,
        best_value=best_evaluation.value,
        best_point=best_evaluation.x,
        slope_range=slope_range,
        boundary_distance=boundary_distance,
    )

In [10]:
class RefinementAction(str, Enum):
    SAMPLE = "sample"
    SPLIT = "split"
    STABLE = "stable"


@dataclass(frozen=True)
class Candidate:
    region_id: int
    action: RefinementAction
    location: float
    source: str

    def __post_init__(self) -> None:
        if self.region_id < 0:
            raise ValueError(
                "Candidate region id must be non-negative."
            )

        if not __import__("math").isfinite(self.location):
            raise ValueError(
                "Candidate location must be finite."
            )

        if not self.source.strip():
            raise ValueError(
                "Candidate source must not be empty."
            )

In [11]:
def generate_sampling_candidates(
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> tuple[Candidate, ...]:
    region = view.region
    evaluations = tuple(
        sorted(
            view.evaluations,
            key=lambda evaluation: evaluation.x,
        )
    )

    candidates: list[Candidate] = []

    def add_candidate(
        location: float,
        source: str,
    ) -> None:
        if not region.contains(
            location,
            tolerances.boundary,
        ):
            return

        if any(
            abs(candidate.location - location)
            <= tolerances.point
            for candidate in candidates
        ):
            return

        if any(
            abs(evaluation.x - location)
            <= tolerances.point
            for evaluation in evaluations
        ):
            return

        candidates.append(
            Candidate(
                region_id=region.id,
                action=RefinementAction.SAMPLE,
                location=location,
                source=source,
            )
        )

    if not evaluations:
        midpoint = (
            region.lower_bound
            + region.upper_bound
        ) / 2.0

        add_candidate(
            midpoint,
            "empty_region_midpoint",
        )

        return tuple(candidates)

    if len(evaluations) == 1:
        point = evaluations[0].x

        left_distance = (
            point - region.lower_bound
        )

        right_distance = (
            region.upper_bound - point
        )

        if left_distance >= right_distance:
            location = (
                region.lower_bound + point
            ) / 2.0

            add_candidate(
                location,
                "single_point_left_gap",
            )

        if right_distance >= left_distance:
            location = (
                point + region.upper_bound
            ) / 2.0

            add_candidate(
                location,
                "single_point_right_gap",
            )

        return tuple(candidates)

    gaps = [
        (
            left.x,
            right.x,
            right.x - left.x,
        )
        for left, right in zip(
            evaluations,
            evaluations[1:],
        )
        if right.x - left.x > tolerances.point
    ]

    if gaps:
        largest_gap = max(
            gap[2]
            for gap in gaps
        )

        for left, right, gap in gaps:
            if (
                abs(gap - largest_gap)
                <= tolerances.point
            ):
                midpoint = (
                    left + right
                ) / 2.0

                add_candidate(
                    midpoint,
                    "largest_coverage_gap",
                )

    left_boundary_midpoint = (
        region.lower_bound
        + evaluations[0].x
    ) / 2.0

    right_boundary_midpoint = (
        evaluations[-1].x
        + region.upper_bound
    ) / 2.0

    add_candidate(
        left_boundary_midpoint,
        "left_boundary_gap",
    )

    add_candidate(
        right_boundary_midpoint,
        "right_boundary_gap",
    )

    return tuple(candidates)

In [12]:
@dataclass(frozen=True)
class SamplingCandidateEvaluation:
    candidate: Candidate
    current_resolution: float
    resulting_resolution: float
    coverage_improvement: float

    @property
    def improves_coverage(self) -> bool:
        return (
            self.resulting_resolution
            < self.current_resolution
        )


def evaluate_sampling_candidate(
    candidate: Candidate,
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> SamplingCandidateEvaluation:
    if candidate.action != RefinementAction.SAMPLE:
        raise ValueError(
            "The candidate must represent a sampling action."
        )

    if candidate.region_id != view.region.id:
        raise ValueError(
            "Candidate region id does not match the view region."
        )

    existing_points = sorted(
        evaluation.x
        for evaluation in view.evaluations
    )

    if any(
        abs(candidate.location - point)
        <= tolerances.point
        for point in existing_points
    ):
        raise ValueError(
            "Sampling candidate is too close to an existing evaluation."
        )

    points_with_candidate = sorted(
        [*existing_points, candidate.location]
    )

    def maximum_gap(points: list[float]) -> float:
        if len(points) < 2:
            return view.region.width

        return max(
            right - left
            for left, right in zip(
                points,
                points[1:],
            )
        )

    current_resolution = maximum_gap(
        existing_points
    )

    resulting_resolution = maximum_gap(
        points_with_candidate
    )

    coverage_improvement = (
        current_resolution
        - resulting_resolution
    )

    return SamplingCandidateEvaluation(
        candidate=candidate,
        current_resolution=current_resolution,
        resulting_resolution=resulting_resolution,
        coverage_improvement=coverage_improvement,
    )

In [13]:
def select_sampling_candidate(
    evaluations: tuple[SamplingCandidateEvaluation, ...],
    tolerances: NumericalTolerances,
) -> SamplingCandidateEvaluation | None:
    valid_evaluations = tuple(
        evaluation
        for evaluation in evaluations
        if evaluation.improves_coverage
        and evaluation.coverage_improvement
        > tolerances.objective
    )

    if not valid_evaluations:
        return None

    return max(
        valid_evaluations,
        key=lambda evaluation: (
            evaluation.coverage_improvement,
            -evaluation.candidate.location,
        ),
    )

In [14]:
@dataclass(frozen=True)
class SplitCandidate:
    region_id: int
    split_location: float
    contraction_factor: float
    left_width: float
    right_width: float

    @property
    def parent_width(self) -> float:
        return self.left_width + self.right_width

    @property
    def satisfies_contraction(self) -> bool:
        return (
            max(
                self.left_width,
                self.right_width,
            )
            <= self.contraction_factor
            * self.parent_width
        )


def create_split_candidate(
    region: Region,
    split_location: float,
    contraction_factor: float,
    tolerance: float,
) -> SplitCandidate:
    if not 0.0 < contraction_factor < 1.0:
        raise ValueError(
            "The contraction factor must satisfy 0 < rho < 1."
        )

    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    if not region.contains(
        split_location,
        tolerance,
    ):
        raise ValueError(
            "Split location must lie inside the region."
        )

    if (
        split_location - region.lower_bound
        <= tolerance
    ):
        raise ValueError(
            "Split location is too close to the lower boundary."
        )

    if (
        region.upper_bound - split_location
        <= tolerance
    ):
        raise ValueError(
            "Split location is too close to the upper boundary."
        )

    left_width = (
        split_location
        - region.lower_bound
    )

    right_width = (
        region.upper_bound
        - split_location
    )

    candidate = SplitCandidate(
        region_id=region.id,
        split_location=split_location,
        contraction_factor=contraction_factor,
        left_width=left_width,
        right_width=right_width,
    )

    if not candidate.satisfies_contraction:
        raise ValueError(
            "The split does not satisfy the contraction guarantee."
        )

    return candidate

In [15]:
def generate_split_locations(
    region: Region,
    contraction_factor: float,
    tolerances: NumericalTolerances,
) -> tuple[float, ...]:
    if not 0.0 < contraction_factor < 1.0:
        raise ValueError(
            "The contraction factor must satisfy 0 < rho < 1."
        )

    if tolerances.point < 0:
        raise ValueError(
            "Point tolerance must be non-negative."
        )

    if contraction_factor < 0.5:
        return tuple()

    lower = region.lower_bound
    upper = region.upper_bound

    left_valid_boundary = (
        contraction_factor * lower
        + (1.0 - contraction_factor) * upper
    )

    right_valid_boundary = (
        (1.0 - contraction_factor) * lower
        + contraction_factor * upper
    )

    midpoint = (lower + upper) / 2.0

    locations = [
        left_valid_boundary,
        midpoint,
        right_valid_boundary,
    ]

    unique_locations: list[float] = []

    for location in locations:
        if not region.contains(
            location,
            tolerances.boundary,
        ):
            continue

        if (
            location - lower
            <= tolerances.point
        ):
            continue

        if (
            upper - location
            <= tolerances.point
        ):
            continue

        if any(
            abs(location - existing)
            <= tolerances.point
            for existing in unique_locations
        ):
            continue

        unique_locations.append(location)

    return tuple(unique_locations)


def generate_split_candidates(
    region: Region,
    contraction_factor: float,
    tolerances: NumericalTolerances,
) -> tuple[SplitCandidate, ...]:
    locations = generate_split_locations(
        region=region,
        contraction_factor=contraction_factor,
        tolerances=tolerances,
    )

    candidates: list[SplitCandidate] = []

    for location in locations:
        candidate = create_split_candidate(
            region=region,
            split_location=location,
            contraction_factor=contraction_factor,
            tolerance=tolerances.point,
        )

        candidates.append(candidate)

    return tuple(candidates)

In [16]:
@dataclass(frozen=True)
class StructuralSplitValue:
    candidate: SplitCandidate
    parent_width: float
    maximum_child_width: float
    structural_improvement: float

    @property
    def relative_improvement(self) -> float:
        if self.parent_width <= 0.0:
            return 0.0

        return (
            self.structural_improvement
            / self.parent_width
        )


def evaluate_structural_split(
    candidate: SplitCandidate,
) -> StructuralSplitValue:
    parent_width = candidate.parent_width

    maximum_child_width = max(
        candidate.left_width,
        candidate.right_width,
    )

    structural_improvement = (
        parent_width
        - maximum_child_width
    )

    return StructuralSplitValue(
        candidate=candidate,
        parent_width=parent_width,
        maximum_child_width=maximum_child_width,
        structural_improvement=structural_improvement,
    )

In [17]:
@dataclass(frozen=True)
class StructuralDifference:
    candidate: SplitCandidate
    left_evaluation_count: int
    right_evaluation_count: int
    value_difference: float | None

    @property
    def is_computable(self) -> bool:
        return self.value_difference is not None


def evaluate_structural_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> StructuralDifference:
    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    if left_evaluations and right_evaluations:
        left_mean = sum(
            evaluation.value
            for evaluation in left_evaluations
        ) / len(left_evaluations)

        right_mean = sum(
            evaluation.value
            for evaluation in right_evaluations
        ) / len(right_evaluations)

        value_difference = abs(
            left_mean - right_mean
        )
    else:
        value_difference = None

    return StructuralDifference(
        candidate=candidate,
        left_evaluation_count=len(left_evaluations),
        right_evaluation_count=len(right_evaluations),
        value_difference=value_difference,
    )

In [18]:
@dataclass(frozen=True)
class DirectionalDifference:
    candidate: SplitCandidate
    left_slope: float | None
    right_slope: float | None
    slope_difference: float | None

    @property
    def is_computable(self) -> bool:
        return self.slope_difference is not None


def estimate_side_slope(
    evaluations: tuple[Evaluation, ...],
    tolerances: NumericalTolerances,
) -> float | None:
    if len(evaluations) < 2:
        return None

    ordered = tuple(
        sorted(
            evaluations,
            key=lambda evaluation: evaluation.x,
        )
    )

    slopes: list[float] = []

    for left, right in zip(
        ordered,
        ordered[1:],
    ):
        dx = right.x - left.x

        if dx <= tolerances.point:
            continue

        slopes.append(
            (right.value - left.value) / dx
        )

    if not slopes:
        return None

    return sum(slopes) / len(slopes)


def evaluate_directional_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> DirectionalDifference:
    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    left_slope = estimate_side_slope(
        left_evaluations,
        tolerances,
    )

    right_slope = estimate_side_slope(
        right_evaluations,
        tolerances,
    )

    if (
        left_slope is not None
        and right_slope is not None
    ):
        slope_difference = abs(
            left_slope - right_slope
        )
    else:
        slope_difference = None

    return DirectionalDifference(
        candidate=candidate,
        left_slope=left_slope,
        right_slope=right_slope,
        slope_difference=slope_difference,
    )

In [19]:
@dataclass(frozen=True)
class SlopeVariationDifference:
    candidate: SplitCandidate
    left_slope_variation: float | None
    right_slope_variation: float | None
    variation_difference: float | None

    @property
    def is_computable(self) -> bool:
        return self.variation_difference is not None


def calculate_slope_variation(
    evaluations: tuple[Evaluation, ...],
    tolerances: NumericalTolerances,
) -> float | None:
    if len(evaluations) < 3:
        return None

    ordered = tuple(
        sorted(
            evaluations,
            key=lambda evaluation: evaluation.x,
        )
    )

    slopes: list[float] = []

    for left, right in zip(
        ordered,
        ordered[1:],
    ):
        dx = right.x - left.x

        if dx <= tolerances.point:
            continue

        slopes.append(
            (right.value - left.value) / dx
        )

    if len(slopes) < 2:
        return None

    return max(slopes) - min(slopes)


def evaluate_slope_variation_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> SlopeVariationDifference:
    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    left_variation = calculate_slope_variation(
        left_evaluations,
        tolerances,
    )

    right_variation = calculate_slope_variation(
        right_evaluations,
        tolerances,
    )

    if (
        left_variation is not None
        and right_variation is not None
    ):
        variation_difference = abs(
            left_variation - right_variation
        )
    else:
        variation_difference = None

    return SlopeVariationDifference(
        candidate=candidate,
        left_slope_variation=left_variation,
        right_slope_variation=right_variation,
        variation_difference=variation_difference,
    )

In [20]:
@dataclass(frozen=True)
class SamplingDensityDifference:
    candidate: SplitCandidate
    left_density: float | None
    right_density: float | None
    density_difference: float | None
    left_coverage_resolution: float | None
    right_coverage_resolution: float | None

    @property
    def is_computable(self) -> bool:
        return self.density_difference is not None


def calculate_side_density(
    evaluations: tuple[Evaluation, ...],
    width: float,
    tolerances: NumericalTolerances,
) -> float | None:
    if width <= tolerances.boundary:
        return None

    return len(evaluations) / width


def calculate_side_coverage_resolution(
    evaluations: tuple[Evaluation, ...],
    lower_bound: float,
    upper_bound: float,
    tolerances: NumericalTolerances,
) -> float | None:
    if upper_bound - lower_bound <= tolerances.boundary:
        return None

    points = sorted(
        evaluation.x
        for evaluation in evaluations
    )

    if not points:
        return upper_bound - lower_bound

    gaps = [
        points[0] - lower_bound,
        upper_bound - points[-1],
    ]

    gaps.extend(
        right - left
        for left, right in zip(
            points,
            points[1:],
        )
    )

    valid_gaps = [
        gap
        for gap in gaps
        if gap >= 0.0
    ]

    return max(valid_gaps) if valid_gaps else 0.0


def evaluate_sampling_density_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> SamplingDensityDifference:
    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    left_density = calculate_side_density(
        left_evaluations,
        candidate.left_width,
        tolerances,
    )

    right_density = calculate_side_density(
        right_evaluations,
        candidate.right_width,
        tolerances,
    )

    if (
        left_density is not None
        and right_density is not None
    ):
        density_difference = abs(
            left_density - right_density
        )
    else:
        density_difference = None

    left_coverage_resolution = (
        calculate_side_coverage_resolution(
            left_evaluations,
            view.region.lower_bound,
            split,
            tolerances,
        )
    )

    right_coverage_resolution = (
        calculate_side_coverage_resolution(
            right_evaluations,
            split,
            view.region.upper_bound,
            tolerances,
        )
    )

    return SamplingDensityDifference(
        candidate=candidate,
        left_density=left_density,
        right_density=right_density,
        density_difference=density_difference,
        left_coverage_resolution=left_coverage_resolution,
        right_coverage_resolution=right_coverage_resolution,
    )

In [21]:
@dataclass(frozen=True)
class UncertaintyDifference:
    candidate: SplitCandidate
    left_uncertainty: float | None
    right_uncertainty: float | None
    uncertainty_difference: float | None

    @property
    def is_computable(self) -> bool:
        return self.uncertainty_difference is not None


def calculate_lipschitz_uncertainty_at_point(
    x: float,
    evaluations: tuple[Evaluation, ...],
    lipschitz_constant: float,
    tolerances: NumericalTolerances,
) -> float | None:
    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    if not evaluations:
        return None

    upper_values = [
        evaluation.value
        + lipschitz_constant
        * abs(x - evaluation.x)
        for evaluation in evaluations
    ]

    lower_values = [
        evaluation.value
        - lipschitz_constant
        * abs(x - evaluation.x)
        for evaluation in evaluations
    ]

    upper_bound = min(upper_values)
    lower_bound = max(lower_values)

    uncertainty = max(
        0.0,
        upper_bound - lower_bound,
    )

    if uncertainty <= tolerances.objective:
        return 0.0

    return uncertainty


def evaluate_uncertainty_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> UncertaintyDifference:
    if lipschitz_constant is None:
        return UncertaintyDifference(
            candidate=candidate,
            left_uncertainty=None,
            right_uncertainty=None,
            uncertainty_difference=None,
        )

    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    left_uncertainty = (
        calculate_lipschitz_uncertainty_at_point(
            split,
            left_evaluations,
            lipschitz_constant,
            tolerances,
        )
    )

    right_uncertainty = (
        calculate_lipschitz_uncertainty_at_point(
            split,
            right_evaluations,
            lipschitz_constant,
            tolerances,
        )
    )

    if (
        left_uncertainty is not None
        and right_uncertainty is not None
    ):
        uncertainty_difference = abs(
            left_uncertainty
            - right_uncertainty
        )
    else:
        uncertainty_difference = None

    return UncertaintyDifference(
        candidate=candidate,
        left_uncertainty=left_uncertainty,
        right_uncertainty=right_uncertainty,
        uncertainty_difference=uncertainty_difference,
    )

In [22]:
@dataclass(frozen=True)
class OptimizationPotentialDifference:
    candidate: SplitCandidate
    left_potential: float | None
    right_potential: float | None
    potential_difference: float | None

    @property
    def is_computable(self) -> bool:
        return self.potential_difference is not None


def calculate_observed_upper_potential(
    evaluations: tuple[Evaluation, ...],
    reference_points: tuple[Evaluation, ...],
    lipschitz_constant: float,
) -> float | None:
    if not evaluations or not reference_points:
        return None

    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    upper_values = []

    for evaluation in evaluations:
        upper_bound = min(
            reference.value
            + lipschitz_constant
            * abs(evaluation.x - reference.x)
            for reference in reference_points
        )

        upper_values.append(upper_bound)

    return max(upper_values)


def evaluate_optimization_potential_difference(
    candidate: SplitCandidate,
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> OptimizationPotentialDifference:
    if lipschitz_constant is None:
        return OptimizationPotentialDifference(
            candidate=candidate,
            left_potential=None,
            right_potential=None,
            potential_difference=None,
        )

    split = candidate.split_location

    left_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x < split - tolerances.point
    )

    right_evaluations = tuple(
        evaluation
        for evaluation in view.evaluations
        if evaluation.x > split + tolerances.point
    )

    left_potential = calculate_observed_upper_potential(
        evaluations=left_evaluations,
        reference_points=view.evaluations,
        lipschitz_constant=lipschitz_constant,
    )

    right_potential = calculate_observed_upper_potential(
        evaluations=right_evaluations,
        reference_points=view.evaluations,
        lipschitz_constant=lipschitz_constant,
    )

    if (
        left_potential is not None
        and right_potential is not None
    ):
        potential_difference = abs(
            left_potential
            - right_potential
        )
    else:
        potential_difference = None

    return OptimizationPotentialDifference(
        candidate=candidate,
        left_potential=left_potential,
        right_potential=right_potential,
        potential_difference=potential_difference,
    )

In [24]:
@dataclass(frozen=True)
class StructuralDifferenceProfile:
    structural_value: StructuralSplitValue
    structural_difference: StructuralDifference
    directional_difference: DirectionalDifference
    slope_variation_difference: SlopeVariationDifference
    sampling_density_difference: SamplingDensityDifference
    uncertainty_difference: UncertaintyDifference
    optimization_potential_difference: OptimizationPotentialDifference

    @property
    def candidate(self) -> SplitCandidate:
        return self.structural_value.candidate

    @property
    def region_id(self) -> int:
        return self.candidate.region_id

    @property
    def split_location(self) -> float:
        return self.candidate.split_location

    @property
    def structural_improvement(self) -> float:
        return self.structural_value.structural_improvement

    @property
    def value_difference(self) -> float | None:
        return self.structural_difference.value_difference

    @property
    def slope_difference(self) -> float | None:
        return self.directional_difference.slope_difference

    @property
    def slope_variation_value(self) -> float | None:
        return (
            self.slope_variation_difference
            .variation_difference
        )

    @property
    def density_difference(self) -> float | None:
        return (
            self.sampling_density_difference
            .density_difference
        )

    @property
    def uncertainty_difference_value(self) -> float | None:
        return (
            self.uncertainty_difference
            .uncertainty_difference
        )

    @property
    def potential_difference(self) -> float | None:
        return (
            self.optimization_potential_difference
            .potential_difference
        )

In [25]:
def dominates_candidate_split(
    first: StructuralDifferenceProfile,
    second: StructuralDifferenceProfile,
    tolerances: NumericalTolerances,
) -> bool:
    if first.region_id != second.region_id:
        raise ValueError(
            "Candidate splits must belong to the same region."
        )

    if tolerances.objective < 0:
        raise ValueError(
            "Objective tolerance must be non-negative."
        )

    comparisons: list[tuple[float, float]] = []

    # Structural improvement: larger is better.
    comparisons.append(
        (
            first.structural_improvement,
            second.structural_improvement,
        )
    )

    # Value difference: larger indicates stronger structural contrast.
    if (
        first.value_difference is not None
        and second.value_difference is not None
    ):
        comparisons.append(
            (
                first.value_difference,
                second.value_difference,
            )
        )

    # Directional difference: larger indicates stronger
    # directional contrast across the split.
    if (
        first.slope_difference is not None
        and second.slope_difference is not None
    ):
        comparisons.append(
            (
                first.slope_difference,
                second.slope_difference,
            )
        )

    # Slope variation difference: larger indicates stronger
    # difference in local slope behavior.
    if (
        first.slope_variation_value is not None
        and second.slope_variation_value is not None
    ):
        comparisons.append(
            (
                first.slope_variation_value,
                second.slope_variation_value,
            )
        )

    # Sampling density difference: larger indicates stronger
    # imbalance in sampling distribution.
    if (
        first.density_difference is not None
        and second.density_difference is not None
    ):
        comparisons.append(
            (
                first.density_difference,
                second.density_difference,
            )
        )

    # Uncertainty difference: larger indicates stronger
    # asymmetry in local certified uncertainty.
    if (
        first.uncertainty_difference_value is not None
        and second.uncertainty_difference_value is not None
    ):
        comparisons.append(
            (
                first.uncertainty_difference_value,
                second.uncertainty_difference_value,
            )
        )

    # Optimization-potential difference: larger indicates
    # stronger difference in local potential.
    if (
        first.potential_difference is not None
        and second.potential_difference is not None
    ):
        comparisons.append(
            (
                first.potential_difference,
                second.potential_difference,
            )
        )

    if not comparisons:
        return False

    at_least_as_good = all(
        first_value
        >= second_value - tolerances.objective
        for first_value, second_value in comparisons
    )

    strictly_better = any(
        first_value
        > second_value + tolerances.objective
        for first_value, second_value in comparisons
    )

    return at_least_as_good and strictly_better

In [26]:
def select_non_dominated_split(
    profiles: tuple[StructuralDifferenceProfile, ...],
    tolerances: NumericalTolerances,
) -> StructuralDifferenceProfile | None:
    if not profiles:
        return None

    region_ids = {
        profile.region_id
        for profile in profiles
    }

    if len(region_ids) != 1:
        raise ValueError(
            "All split profiles must belong to the same region."
        )

    non_dominated: list[StructuralDifferenceProfile] = []

    for candidate in profiles:
        is_dominated = False

        for other in profiles:
            if other is candidate:
                continue

            if dominates_candidate_split(
                first=other,
                second=candidate,
                tolerances=tolerances,
            ):
                is_dominated = True
                break

        if not is_dominated:
            non_dominated.append(candidate)

    if not non_dominated:
        return None

    return max(
        non_dominated,
        key=lambda profile: (
            profile.structural_improvement,
            -profile.split_location,
        ),
    )

In [27]:
@dataclass(frozen=True)
class RegionRefinementObjective:
    region_id: int
    coverage_resolution: float
    behavior_resolution: float
    uncertainty_resolution: float | None
    potential_resolution: float | None

    @property
    def has_unresolved_coverage(self) -> bool:
        return self.coverage_resolution > 0.0

    @property
    def has_unresolved_behavior(self) -> bool:
        return self.behavior_resolution > 0.0

    @property
    def has_unresolved_uncertainty(self) -> bool:
        return (
            self.uncertainty_resolution is not None
            and self.uncertainty_resolution > 0.0
        )

    @property
    def has_unresolved_potential(self) -> bool:
        return (
            self.potential_resolution is not None
            and self.potential_resolution > 0.0
        )

    @property
    def has_unresolved_information(self) -> bool:
        return (
            self.has_unresolved_coverage
            or self.has_unresolved_behavior
            or self.has_unresolved_uncertainty
            or self.has_unresolved_potential
        )

In [28]:
@dataclass(frozen=True)
class CoverageResolutionObjective:
    region_id: int
    resolution: float
    region_width: float

    @property
    def normalized_resolution(self) -> float:
        if self.region_width <= 0.0:
            return 0.0

        return (
            self.resolution
            / self.region_width
        )

    @property
    def is_unresolved(self) -> bool:
        return self.resolution > 0.0


def calculate_coverage_resolution(
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> float:
    region = view.region

    if not view.evaluations:
        return region.width

    points = sorted(
        evaluation.x
        for evaluation in view.evaluations
    )

    gaps = [
        points[0] - region.lower_bound,
        region.upper_bound - points[-1],
    ]

    gaps.extend(
        right - left
        for left, right in zip(
            points,
            points[1:],
        )
        if right - left > tolerances.point
    )

    valid_gaps = [
        gap
        for gap in gaps
        if gap >= 0.0
    ]

    if not valid_gaps:
        return 0.0

    return max(valid_gaps)


def evaluate_coverage_resolution_objective(
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> CoverageResolutionObjective:
    resolution = calculate_coverage_resolution(
        view=view,
        tolerances=tolerances,
    )

    return CoverageResolutionObjective(
        region_id=view.region.id,
        resolution=resolution,
        region_width=view.region.width,
    )

In [29]:
@dataclass(frozen=True)
class BehaviorResolutionObjective:
    region_id: int
    slope_variation: float | None
    evaluation_count: int

    @property
    def is_computable(self) -> bool:
        return self.slope_variation is not None

    @property
    def resolution(self) -> float | None:
        return self.slope_variation

    @property
    def is_unresolved(self) -> bool:
        return (
            self.slope_variation is not None
            and self.slope_variation > 0.0
        )


def evaluate_behavior_resolution_objective(
    view: RegionEvaluationView,
    tolerances: NumericalTolerances,
) -> BehaviorResolutionObjective:
    slope_variation = calculate_slope_variation(
        evaluations=view.evaluations,
        tolerances=tolerances,
    )

    return BehaviorResolutionObjective(
        region_id=view.region.id,
        slope_variation=slope_variation,
        evaluation_count=view.count,
    )

In [30]:
@dataclass(frozen=True)
class UncertaintyResolutionObjective:
    region_id: int
    uncertainty: float | None
    lipschitz_constant: float | None

    @property
    def is_computable(self) -> bool:
        return (
            self.uncertainty is not None
            and self.lipschitz_constant is not None
        )

    @property
    def resolution(self) -> float | None:
        return self.uncertainty

    @property
    def is_unresolved(self) -> bool:
        return (
            self.uncertainty is not None
            and self.uncertainty > 0.0
        )


def calculate_observed_uncertainty(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> float | None:
    if lipschitz_constant is None:
        return None

    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    if not view.evaluations:
        return None

    uncertainties: list[float] = []

    for evaluation in view.evaluations:
        uncertainty = calculate_lipschitz_uncertainty_at_point(
            x=evaluation.x,
            evaluations=view.evaluations,
            lipschitz_constant=lipschitz_constant,
            tolerances=tolerances,
        )

        if uncertainty is not None:
            uncertainties.append(uncertainty)

    if not uncertainties:
        return None

    return max(uncertainties)


def evaluate_uncertainty_resolution_objective(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> UncertaintyResolutionObjective:
    uncertainty = calculate_observed_uncertainty(
        view=view,
        lipschitz_constant=lipschitz_constant,
        tolerances=tolerances,
    )

    return UncertaintyResolutionObjective(
        region_id=view.region.id,
        uncertainty=uncertainty,
        lipschitz_constant=lipschitz_constant,
    )

In [31]:
@dataclass(frozen=True)
class OptimizationPotentialResolutionObjective:
    region_id: int
    potential: float | None
    lipschitz_constant: float | None

    @property
    def is_computable(self) -> bool:
        return (
            self.potential is not None
            and self.lipschitz_constant is not None
        )

    @property
    def resolution(self) -> float | None:
        return self.potential

    @property
    def is_unresolved(self) -> bool:
        return (
            self.potential is not None
            and self.potential > 0.0
        )


def calculate_observed_optimization_potential(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
) -> float | None:
    if lipschitz_constant is None:
        return None

    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    if not view.evaluations:
        return None

    potential = calculate_observed_upper_potential(
        evaluations=view.evaluations,
        reference_points=view.evaluations,
        lipschitz_constant=lipschitz_constant,
    )

    return potential


def evaluate_optimization_potential_resolution_objective(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
) -> OptimizationPotentialResolutionObjective:
    potential = calculate_observed_optimization_potential(
        view=view,
        lipschitz_constant=lipschitz_constant,
    )

    return OptimizationPotentialResolutionObjective(
        region_id=view.region.id,
        potential=potential,
        lipschitz_constant=lipschitz_constant,
    )

In [32]:
@dataclass(frozen=True)
class UnresolvedInformationState:
    region_id: int
    coverage: CoverageResolutionObjective
    behavior: BehaviorResolutionObjective
    uncertainty: UncertaintyResolutionObjective
    potential: OptimizationPotentialResolutionObjective

    @property
    def has_unresolved_coverage(self) -> bool:
        return self.coverage.is_unresolved

    @property
    def has_unresolved_behavior(self) -> bool:
        return self.behavior.is_unresolved

    @property
    def has_unresolved_uncertainty(self) -> bool:
        return self.uncertainty.is_unresolved

    @property
    def has_unresolved_potential(self) -> bool:
        return self.potential.is_unresolved

    @property
    def has_unresolved_information(self) -> bool:
        return (
            self.has_unresolved_coverage
            or self.has_unresolved_behavior
            or self.has_unresolved_uncertainty
            or self.has_unresolved_potential
        )

    @property
    def unresolved_dimensions(self) -> tuple[str, ...]:
        dimensions: list[str] = []

        if self.has_unresolved_coverage:
            dimensions.append("coverage")

        if self.has_unresolved_behavior:
            dimensions.append("behavior")

        if self.has_unresolved_uncertainty:
            dimensions.append("uncertainty")

        if self.has_unresolved_potential:
            dimensions.append("potential")

        return tuple(dimensions)


def build_unresolved_information_state(
    coverage: CoverageResolutionObjective,
    behavior: BehaviorResolutionObjective,
    uncertainty: UncertaintyResolutionObjective,
    potential: OptimizationPotentialResolutionObjective,
) -> UnresolvedInformationState:
    region_ids = {
        coverage.region_id,
        behavior.region_id,
        uncertainty.region_id,
        potential.region_id,
    }

    if len(region_ids) != 1:
        raise ValueError(
            "All resolution objectives must belong to the same region."
        )

    return UnresolvedInformationState(
        region_id=coverage.region_id,
        coverage=coverage,
        behavior=behavior,
        uncertainty=uncertainty,
        potential=potential,
    )

In [33]:
@dataclass(frozen=True)
class ResolutionCriteria:
    coverage_tolerance: float
    behavior_tolerance: float
    uncertainty_tolerance: float
    potential_tolerance: float

    def __post_init__(self) -> None:
        values = {
            "coverage_tolerance": self.coverage_tolerance,
            "behavior_tolerance": self.behavior_tolerance,
            "uncertainty_tolerance": self.uncertainty_tolerance,
            "potential_tolerance": self.potential_tolerance,
        }

        for name, value in values.items():
            if value < 0.0:
                raise ValueError(
                    f"Resolution tolerance '{name}' "
                    "must be non-negative."
                )


def is_coverage_resolved(
    objective: CoverageResolutionObjective,
    criteria: ResolutionCriteria,
) -> bool:
    return (
        objective.resolution
        <= criteria.coverage_tolerance
    )


def is_behavior_resolved(
    objective: BehaviorResolutionObjective,
    criteria: ResolutionCriteria,
) -> bool:
    if objective.resolution is None:
        return False

    return (
        objective.resolution
        <= criteria.behavior_tolerance
    )


def is_uncertainty_resolved(
    objective: UncertaintyResolutionObjective,
    criteria: ResolutionCriteria,
) -> bool:
    if objective.resolution is None:
        return False

    return (
        objective.resolution
        <= criteria.uncertainty_tolerance
    )


def is_potential_resolved(
    objective: OptimizationPotentialResolutionObjective,
    criteria: ResolutionCriteria,
) -> bool:
    if objective.resolution is None:
        return False

    return (
        objective.resolution
        <= criteria.potential_tolerance
    )

In [34]:
@dataclass(frozen=True)
class DecisionStability:
    region_id: int
    coverage_resolved: bool
    behavior_resolved: bool
    uncertainty_resolved: bool
    potential_resolved: bool
    uncertainty_available: bool
    potential_available: bool

    @property
    def is_stable(self) -> bool:
        if not self.coverage_resolved:
            return False

        if not self.behavior_resolved:
            return False

        if self.uncertainty_available:
            if not self.uncertainty_resolved:
                return False

        if self.potential_available:
            if not self.potential_resolved:
                return False

        return True


def evaluate_decision_stability(
    state: UnresolvedInformationState,
    criteria: ResolutionCriteria,
) -> DecisionStability:
    coverage_resolved = is_coverage_resolved(
        objective=state.coverage,
        criteria=criteria,
    )

    behavior_resolved = is_behavior_resolved(
        objective=state.behavior,
        criteria=criteria,
    )

    uncertainty_available = (
        state.uncertainty.resolution is not None
    )

    uncertainty_resolved = (
        is_uncertainty_resolved(
            objective=state.uncertainty,
            criteria=criteria,
        )
        if uncertainty_available
        else False
    )

    potential_available = (
        state.potential.resolution is not None
    )

    potential_resolved = (
        is_potential_resolved(
            objective=state.potential,
            criteria=criteria,
        )
        if potential_available
        else False
    )

    return DecisionStability(
        region_id=state.region_id,
        coverage_resolved=coverage_resolved,
        behavior_resolved=behavior_resolved,
        uncertainty_resolved=uncertainty_resolved,
        potential_resolved=potential_resolved,
        uncertainty_available=uncertainty_available,
        potential_available=potential_available,
    )

In [35]:
def select_refinement_action(
    view: RegionEvaluationView,
    analysis: RegionAnalysis,
    stability: DecisionStability,
    sampling_candidate: SamplingCandidateEvaluation | None,
    split_candidate: StructuralDifferenceProfile | None,
) -> RefinementAction:
    if stability.is_stable:
        return RefinementAction.STABLE

    if (
        sampling_candidate is not None
        and sampling_candidate.improves_coverage
    ):
        sampling_improvement = (
            sampling_candidate.coverage_improvement
        )
    else:
        sampling_improvement = 0.0

    if split_candidate is not None:
        split_improvement = (
            split_candidate.structural_improvement
        )
    else:
        split_improvement = 0.0

    if sampling_candidate is None and split_candidate is None:
        return RefinementAction.STABLE

    if sampling_improvement > split_improvement:
        return RefinementAction.SAMPLE

    if split_improvement > sampling_improvement:
        return RefinementAction.SPLIT

    if (
        analysis.coverage_resolution
        > analysis.width * 0.5
    ):
        if sampling_candidate is not None:
            return RefinementAction.SAMPLE

    if split_candidate is not None:
        return RefinementAction.SPLIT

    if sampling_candidate is not None:
        return RefinementAction.SAMPLE

    return RefinementAction.STABLE

In [36]:
def select_global_region(
    regions: tuple[Region, ...],
    analyses: dict[int, RegionAnalysis],
    actions: dict[int, RefinementAction],
) -> Region | None:
    eligible_regions = tuple(
        region
        for region in regions
        if region.state == RegionState.ACTIVE
        and actions.get(region.id) != RefinementAction.STABLE
    )

    if not eligible_regions:
        return None

    for region in eligible_regions:
        if region.id not in analyses:
            raise KeyError(
                f"Analysis for region id={region.id} was not found."
            )

    return max(
        eligible_regions,
        key=lambda region: (
            analyses[region.id].width,
            -region.id,
        ),
    )

In [37]:
@dataclass(frozen=True)
class GlobalRegionPriority:
    region_id: int
    unresolved_dimension_count: int
    potential: float | None
    coverage_resolution: float
    behavior_resolution: float | None
    region_width: float

    @property
    def priority_key(self) -> tuple:
        potential_value = (
            self.potential
            if self.potential is not None
            else float("-inf")
        )

        behavior_value = (
            self.behavior_resolution
            if self.behavior_resolution is not None
            else float("-inf")
        )

        return (
            self.unresolved_dimension_count,
            potential_value,
            self.coverage_resolution,
            behavior_value,
            self.region_width,
            -self.region_id,
        )


def calculate_global_region_priority(
    state: UnresolvedInformationState,
) -> GlobalRegionPriority:
    return GlobalRegionPriority(
        region_id=state.region_id,
        unresolved_dimension_count=len(
            state.unresolved_dimensions
        ),
        potential=state.potential.resolution,
        coverage_resolution=state.coverage.resolution,
        behavior_resolution=state.behavior.resolution,
        region_width=state.coverage.region_width,
    )


def select_highest_priority_region(
    priorities: tuple[GlobalRegionPriority, ...],
) -> GlobalRegionPriority | None:
    if not priorities:
        return None

    return max(
        priorities,
        key=lambda priority: priority.priority_key,
    )

In [38]:
@dataclass(frozen=True)
class GlobalOptimizationObjective:
    best_point: float | None
    best_value: float | None
    evaluation_count: int

    @property
    def has_incumbent(self) -> bool:
        return (
            self.best_point is not None
            and self.best_value is not None
        )

    @property
    def is_initialized(self) -> bool:
        return self.has_incumbent

    def update(
        self,
        evaluation: Evaluation,
        tolerances: NumericalTolerances,
    ) -> "GlobalOptimizationObjective":
        if evaluation.value > (
            self.best_value
            if self.best_value is not None
            else float("-inf")
        ) + tolerances.objective:
            return GlobalOptimizationObjective(
                best_point=evaluation.x,
                best_value=evaluation.value,
                evaluation_count=self.evaluation_count + 1,
            )

        return GlobalOptimizationObjective(
            best_point=self.best_point,
            best_value=self.best_value,
            evaluation_count=self.evaluation_count + 1,
        )


def build_global_optimization_objective(
    history: EvaluationHistory,
) -> GlobalOptimizationObjective:
    if not history.evaluations:
        return GlobalOptimizationObjective(
            best_point=None,
            best_value=None,
            evaluation_count=0,
        )

    best_evaluation = max(
        history.evaluations,
        key=lambda evaluation: evaluation.value,
    )

    return GlobalOptimizationObjective(
        best_point=best_evaluation.x,
        best_value=best_evaluation.value,
        evaluation_count=history.count,
    )

In [39]:
def update_global_incumbent(
    current: GlobalOptimizationObjective,
    evaluation: Evaluation,
    tolerances: NumericalTolerances,
) -> GlobalOptimizationObjective:
    if tolerances.objective < 0.0:
        raise ValueError(
            "Objective tolerance must be non-negative."
        )

    if (
        current.best_value is None
        or evaluation.value
        > current.best_value + tolerances.objective
    ):
        best_point = evaluation.x
        best_value = evaluation.value
    else:
        best_point = current.best_point
        best_value = current.best_value

    return GlobalOptimizationObjective(
        best_point=best_point,
        best_value=best_value,
        evaluation_count=current.evaluation_count + 1,
    )


def initialize_global_incumbent(
    history: EvaluationHistory,
) -> GlobalOptimizationObjective:
    return build_global_optimization_objective(
        history=history,
    )


def get_global_best_evaluation(
    history: EvaluationHistory,
) -> Evaluation | None:
    if not history.evaluations:
        return None

    return max(
        history.evaluations,
        key=lambda evaluation: evaluation.value,
    )

In [40]:
@dataclass
class ARRGOGlobalState:
    config: ARRGOConfig
    history: EvaluationHistory
    hierarchy: RegionHierarchy
    incumbent: GlobalOptimizationObjective

    @property
    def evaluation_count(self) -> int:
        return self.history.count

    @property
    def best_point(self) -> float | None:
        return self.incumbent.best_point

    @property
    def best_value(self) -> float | None:
        return self.incumbent.best_value

    @property
    def active_regions(self) -> tuple[Region, ...]:
        return self.hierarchy.active_regions()

    @property
    def budget_exhausted(self) -> bool:
        return (
            self.evaluation_count
            >= self.config.max_evaluations
        )


def create_initial_global_state(
    config: ARRGOConfig,
) -> ARRGOGlobalState:
    history = EvaluationHistory(
        point_tolerance=config.tolerances.point,
    )

    hierarchy = RegionHierarchy()

    root_region = Region(
        id=0,
        lower_bound=config.lower_bound,
        upper_bound=config.upper_bound,
    )

    hierarchy.add(root_region)

    incumbent = GlobalOptimizationObjective(
        best_point=None,
        best_value=None,
        evaluation_count=0,
    )

    return ARRGOGlobalState(
        config=config,
        history=history,
        hierarchy=hierarchy,
        incumbent=incumbent,
    )


def update_global_state_after_evaluation(
    state: ARRGOGlobalState,
    evaluation: Evaluation,
) -> None:
    state.incumbent = update_global_incumbent(
        current=state.incumbent,
        evaluation=evaluation,
        tolerances=state.config.tolerances,
    )

In [41]:
def get_shared_evaluations_for_region(
    region: Region,
    history: EvaluationHistory,
    tolerance: float,
) -> tuple[Evaluation, ...]:
    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    return history.values_in_interval(
        lower_bound=region.lower_bound,
        upper_bound=region.upper_bound,
    )


def build_shared_region_view(
    region: Region,
    history: EvaluationHistory,
    tolerance: float,
) -> RegionEvaluationView:
    evaluations = get_shared_evaluations_for_region(
        region=region,
        history=history,
        tolerance=tolerance,
    )

    filtered_evaluations = tuple(
        evaluation
        for evaluation in evaluations
        if region.contains(
            evaluation.x,
            tolerance,
        )
    )

    return RegionEvaluationView(
        region=region,
        evaluations=filtered_evaluations,
    )


def shared_evaluation_ids(
    regions: tuple[Region, ...],
    history: EvaluationHistory,
    tolerance: float,
) -> dict[int, tuple[int, ...]]:
    return {
        region.id: tuple(
            evaluation.id
            for evaluation in get_shared_evaluations_for_region(
                region=region,
                history=history,
                tolerance=tolerance,
            )
        )
        for region in regions
    }

In [42]:
def create_child_regions_from_split(
    hierarchy: RegionHierarchy,
    split_candidate: SplitCandidate,
) -> tuple[Region, Region]:
    parent = hierarchy.get(
        split_candidate.region_id
    )

    if parent.state == RegionState.REFINED:
        raise ValueError(
            "A refined region cannot be split again."
        )

    if parent.children_ids:
        raise ValueError(
            "The region already has child regions."
        )

    split = split_candidate.split_location

    if not parent.contains(
        split,
        tolerance=0.0,
    ):
        raise ValueError(
            "Split location must lie inside the parent region."
        )

    left_child_id = max(
        (region.id for region in hierarchy.regions),
        default=-1,
    ) + 1

    right_child_id = left_child_id + 1

    left_child = Region(
        id=left_child_id,
        lower_bound=parent.lower_bound,
        upper_bound=split,
        parent_id=parent.id,
    )

    right_child = Region(
        id=right_child_id,
        lower_bound=split,
        upper_bound=parent.upper_bound,
        parent_id=parent.id,
    )

    if (
        left_child.lower_bound != parent.lower_bound
        or right_child.upper_bound != parent.upper_bound
    ):
        raise RuntimeError(
            "Child regions do not cover the parent boundaries."
        )

    if left_child.upper_bound != right_child.lower_bound:
        raise RuntimeError(
            "Child regions do not meet at the split location."
        )

    hierarchy.add(left_child)
    hierarchy.add(right_child)

    parent.mark_refined()

    return left_child, right_child


def validate_region_hierarchy(
    hierarchy: RegionHierarchy,
) -> None:
    for region in hierarchy.regions:
        if region.parent_id is None:
            continue

        parent = hierarchy.get(region.parent_id)

        if region.id not in parent.children_ids:
            raise RuntimeError(
                f"Region {region.id} is missing from "
                f"parent {parent.id} children."
            )

        if not (
            parent.lower_bound
            <= region.lower_bound
            < region.upper_bound
            <= parent.upper_bound
        ):
            raise RuntimeError(
                f"Region {region.id} lies outside "
                f"parent {parent.id}."
            )

    for region in hierarchy.regions:
        children = hierarchy.children_of(region.id)

        if not children:
            continue

        if len(children) != 2:
            raise RuntimeError(
                f"Region {region.id} must have exactly "
                "two children after binary refinement."
            )

        left_child, right_child = sorted(
            children,
            key=lambda child: child.lower_bound,
        )

        if left_child.upper_bound != right_child.lower_bound:
            raise RuntimeError(
                f"Children of region {region.id} "
                "do not form a continuous partition."
            )

        if left_child.lower_bound != region.lower_bound:
            raise RuntimeError(
                f"Left child of region {region.id} "
                "does not start at the parent boundary."
            )

        if right_child.upper_bound != region.upper_bound:
            raise RuntimeError(
                f"Right child of region {region.id} "
                "does not end at the parent boundary."
            )

In [43]:
def apply_refinement_action(
    region: Region,
    action: RefinementAction,
) -> None:
    if action == RefinementAction.SAMPLE:
        if region.state != RegionState.ACTIVE:
            raise ValueError(
                "Sampling can only be applied to an active region."
            )

        region.activate()
        return

    if action == RefinementAction.SPLIT:
        if region.state != RegionState.ACTIVE:
            raise ValueError(
                "Splitting can only be applied to an active region."
            )

        region.mark_refined()
        return

    if action == RefinementAction.STABLE:
        if region.state == RegionState.REFINED:
            raise ValueError(
                "A refined region cannot become stable."
            )

        region.mark_stable()
        return

    raise ValueError(
        f"Unsupported refinement action: {action}"
    )


def activate_children_after_split(
    hierarchy: RegionHierarchy,
    parent_id: int,
) -> tuple[Region, ...]:
    parent = hierarchy.get(parent_id)

    if parent.state != RegionState.REFINED:
        raise ValueError(
            "Only a refined region can activate its children."
        )

    children = hierarchy.children_of(parent_id)

    if not children:
        raise ValueError(
            "A refined region must have child regions."
        )

    for child in children:
        child.activate()

    return children


def get_active_leaf_regions(
    hierarchy: RegionHierarchy,
) -> tuple[Region, ...]:
    return tuple(
        region
        for region in hierarchy.active_regions()
        if not region.children_ids
    )

In [44]:
@dataclass(frozen=True)
class TerminationStatus:
    terminated: bool
    reason: TerminationReason | None
    message: str


def check_budget_termination(
    state: ARRGOGlobalState,
) -> TerminationStatus | None:
    if state.budget_exhausted:
        return TerminationStatus(
            terminated=True,
            reason=TerminationReason.BUDGET_EXHAUSTED,
            message="The maximum evaluation budget has been exhausted.",
        )

    return None


def check_certified_termination(
    state: ARRGOGlobalState,
    global_gap: float | None,
) -> TerminationStatus | None:
    if state.config.mode != ARRGOExecutionMode.CERTIFIED:
        return None

    if global_gap is None:
        return None

    if global_gap <= state.config.optimization_tolerance:
        return TerminationStatus(
            terminated=True,
            reason=TerminationReason.CERTIFIED_TOLERANCE,
            message=(
                "The certified global optimality gap "
                "is within the requested tolerance."
            ),
        )

    return None


def check_no_action_termination(
    actions: dict[int, RefinementAction],
) -> TerminationStatus | None:
    valid_actions = tuple(
        action
        for action in actions.values()
        if action != RefinementAction.STABLE
    )

    if valid_actions:
        return None

    return TerminationStatus(
        terminated=True,
        reason=TerminationReason.NO_VALID_ACTION,
        message="No valid refinement action remains.",
    )


def evaluate_termination(
    state: ARRGOGlobalState,
    global_gap: float | None,
    actions: dict[int, RefinementAction],
) -> TerminationStatus:
    budget_status = check_budget_termination(
        state=state,
    )

    if budget_status is not None:
        return budget_status

    certified_status = check_certified_termination(
        state=state,
        global_gap=global_gap,
    )

    if certified_status is not None:
        return certified_status

    no_action_status = check_no_action_termination(
        actions=actions,
    )

    if no_action_status is not None:
        return no_action_status

    return TerminationStatus(
        terminated=False,
        reason=None,
        message="ARRGO can continue execution.",
    )

In [45]:
def contraction_bound(
    region: Region,
    contraction_factor: float,
) -> float:
    if not 0.0 < contraction_factor < 1.0:
        raise ValueError(
            "The contraction factor must satisfy 0 < rho < 1."
        )

    return (
        contraction_factor
        * region.width
    )


def satisfies_contraction_guarantee(
    region: Region,
    split_location: float,
    contraction_factor: float,
    tolerance: float,
) -> bool:
    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    if not 0.0 < contraction_factor < 1.0:
        raise ValueError(
            "The contraction factor must satisfy 0 < rho < 1."
        )

    left_width = (
        split_location
        - region.lower_bound
    )

    right_width = (
        region.upper_bound
        - split_location
    )

    maximum_child_width = max(
        left_width,
        right_width,
    )

    allowed_width = (
        contraction_factor
        * region.width
    )

    return (
        maximum_child_width
        <= allowed_width + tolerance
    )


def validate_split_contraction(
    region: Region,
    split_candidate: SplitCandidate,
    tolerance: float,
) -> None:
    if split_candidate.region_id != region.id:
        raise ValueError(
            "Split candidate does not belong to the given region."
        )

    if not satisfies_contraction_guarantee(
        region=region,
        split_location=split_candidate.split_location,
        contraction_factor=split_candidate.contraction_factor,
        tolerance=tolerance,
    ):
        raise ValueError(
            "The split violates the contraction guarantee."
        )

In [46]:
@dataclass(frozen=True)
class ConvergenceState:
    iteration: int
    evaluation_count: int
    active_region_count: int
    maximum_active_width: float
    best_value: float | None

    @property
    def has_evaluations(self) -> bool:
        return self.evaluation_count > 0

    @property
    def has_active_regions(self) -> bool:
        return self.active_region_count > 0


def build_convergence_state(
    state: ARRGOGlobalState,
) -> ConvergenceState:
    active_regions = state.active_regions

    if active_regions:
        maximum_active_width = max(
            region.width
            for region in active_regions
        )
    else:
        maximum_active_width = 0.0

    return ConvergenceState(
        iteration=0,
        evaluation_count=state.evaluation_count,
        active_region_count=len(active_regions),
        maximum_active_width=maximum_active_width,
        best_value=state.best_value,
    )


def verify_spatial_contraction(
    parent: Region,
    children: tuple[Region, ...],
    contraction_factor: float,
    tolerance: float,
) -> bool:
    if not children:
        return False

    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    maximum_child_width = max(
        child.width
        for child in children
    )

    return (
        maximum_child_width
        <= contraction_factor * parent.width
        + tolerance
    )


def verify_information_monotonicity(
    previous_history: EvaluationHistory,
    current_history: EvaluationHistory,
) -> bool:
    previous_ids = {
        evaluation.id
        for evaluation in previous_history.evaluations
    }

    current_ids = {
        evaluation.id
        for evaluation in current_history.evaluations
    }

    return previous_ids.issubset(current_ids)


def verify_global_incumbent_monotonicity(
    previous_best_value: float | None,
    current_best_value: float | None,
    tolerance: float,
) -> bool:
    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    if previous_best_value is None:
        return True

    if current_best_value is None:
        return False

    return (
        current_best_value
        >= previous_best_value - tolerance
    )

In [47]:
@dataclass(frozen=True)
class ExactRegionalUncertainty:
    region_id: int
    maximum_uncertainty: float
    maximizer_points: tuple[float, ...]
    lipschitz_constant: float

    @property
    def is_zero(self) -> bool:
        return self.maximum_uncertainty == 0.0


def calculate_lipschitz_upper_bound(
    x: float,
    evaluations: tuple[Evaluation, ...],
    lipschitz_constant: float,
) -> float:
    return min(
        evaluation.value
        + lipschitz_constant
        * abs(x - evaluation.x)
        for evaluation in evaluations
    )


def calculate_lipschitz_lower_bound(
    x: float,
    evaluations: tuple[Evaluation, ...],
    lipschitz_constant: float,
) -> float:
    return max(
        evaluation.value
        - lipschitz_constant
        * abs(x - evaluation.x)
        for evaluation in evaluations
    )


def calculate_lipschitz_uncertainty(
    x: float,
    evaluations: tuple[Evaluation, ...],
    lipschitz_constant: float,
) -> float:
    upper_bound = calculate_lipschitz_upper_bound(
        x=x,
        evaluations=evaluations,
        lipschitz_constant=lipschitz_constant,
    )

    lower_bound = calculate_lipschitz_lower_bound(
        x=x,
        evaluations=evaluations,
        lipschitz_constant=lipschitz_constant,
    )

    return max(
        0.0,
        upper_bound - lower_bound,
    )


def _envelope_intersections_on_interval(
    evaluations: tuple[Evaluation, ...],
    lipschitz_constant: float,
    lower_bound: float,
    upper_bound: float,
) -> tuple[float, ...]:
    candidates: list[float] = []

    midpoint = (
        lower_bound + upper_bound
    ) / 2.0

    for first_index, first in enumerate(evaluations):
        first_slope = (
            lipschitz_constant
            if midpoint >= first.x
            else -lipschitz_constant
        )

        first_intercept = (
            first.value
            - first_slope * first.x
        )

        for second in evaluations[first_index + 1:]:
            second_slope = (
                lipschitz_constant
                if midpoint >= second.x
                else -lipschitz_constant
            )

            second_intercept = (
                second.value
                - second_slope * second.x
            )

            slope_difference = (
                first_slope - second_slope
            )

            if abs(slope_difference) <= 1e-15:
                continue

            intersection = (
                second_intercept
                - first_intercept
            ) / slope_difference

            if (
                lower_bound
                <= intersection
                <= upper_bound
            ):
                candidates.append(intersection)

    return tuple(candidates)


def calculate_exact_regional_uncertainty(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> ExactRegionalUncertainty | None:
    if lipschitz_constant is None:
        return None

    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    region = view.region
    evaluations = view.evaluations

    if not evaluations:
        return None

    breakpoints = {
        region.lower_bound,
        region.upper_bound,
    }

    breakpoints.update(
        evaluation.x
        for evaluation in evaluations
    )

    ordered_breakpoints = sorted(
        breakpoints
    )

    for left, right in zip(
        ordered_breakpoints,
        ordered_breakpoints[1:],
    ):
        if right - left <= tolerances.point:
            continue

        breakpoints.update(
            _envelope_intersections_on_interval(
                evaluations=evaluations,
                lipschitz_constant=lipschitz_constant,
                lower_bound=left,
                upper_bound=right,
            )
        )

    candidate_points = sorted(
        point
        for point in breakpoints
        if (
            region.lower_bound
            - tolerances.boundary
            <= point
            <= region.upper_bound
            + tolerances.boundary
        )
    )

    uncertainty_values = [
        (
            point,
            calculate_lipschitz_uncertainty(
                x=point,
                evaluations=evaluations,
                lipschitz_constant=lipschitz_constant,
            ),
        )
        for point in candidate_points
    ]

    maximum_uncertainty = max(
        value
        for _, value in uncertainty_values
    )

    maximizer_points = tuple(
        point
        for point, value in uncertainty_values
        if abs(
            value - maximum_uncertainty
        ) <= tolerances.objective
    )

    if maximum_uncertainty <= tolerances.objective:
        maximum_uncertainty = 0.0

    return ExactRegionalUncertainty(
        region_id=region.id,
        maximum_uncertainty=maximum_uncertainty,
        maximizer_points=maximizer_points,
        lipschitz_constant=lipschitz_constant,
    )

In [48]:
@dataclass(frozen=True)
class ExactRegionalPotential:
    region_id: int
    maximum_potential: float
    maximizer_points: tuple[float, ...]
    lipschitz_constant: float

    @property
    def is_computable(self) -> bool:
        return True


def calculate_exact_regional_potential(
    view: RegionEvaluationView,
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> ExactRegionalPotential | None:
    if lipschitz_constant is None:
        return None

    if lipschitz_constant <= 0.0:
        raise ValueError(
            "The Lipschitz constant must be positive."
        )

    region = view.region
    evaluations = view.evaluations

    if not evaluations:
        return None

    candidate_points: set[float] = {
        region.lower_bound,
        region.upper_bound,
    }

    candidate_points.update(
        evaluation.x
        for evaluation in evaluations
    )

    ordered_points = sorted(
        evaluation.x
        for evaluation in evaluations
    )

    for left_x, right_x in zip(
        ordered_points,
        ordered_points[1:],
    ):
        if right_x - left_x <= tolerances.point:
            continue

        midpoint = (
            left_x + right_x
        ) / 2.0

        active_evaluations = tuple(
            evaluation
            for evaluation in evaluations
            if abs(
                evaluation.x - midpoint
            ) <= (
                (right_x - left_x) / 2.0
            )
        )

        for first_index, first in enumerate(
            active_evaluations
        ):
            first_slope = (
                lipschitz_constant
                if midpoint >= first.x
                else -lipschitz_constant
            )

            first_intercept = (
                first.value
                - first_slope * first.x
            )

            for second in active_evaluations[
                first_index + 1:
            ]:
                second_slope = (
                    lipschitz_constant
                    if midpoint >= second.x
                    else -lipschitz_constant
                )

                second_intercept = (
                    second.value
                    - second_slope * second.x
                )

                slope_difference = (
                    first_slope - second_slope
                )

                if abs(slope_difference) <= tolerances.slope:
                    continue

                intersection = (
                    second_intercept
                    - first_intercept
                ) / slope_difference

                if (
                    left_x
                    <= intersection
                    <= right_x
                ):
                    candidate_points.add(
                        intersection
                    )

    valid_points = sorted(
        point
        for point in candidate_points
        if (
            region.lower_bound
            - tolerances.boundary
            <= point
            <= region.upper_bound
            + tolerances.boundary
        )
    )

    potential_values = [
        (
            point,
            calculate_lipschitz_upper_bound(
                x=point,
                evaluations=evaluations,
                lipschitz_constant=lipschitz_constant,
            ),
        )
        for point in valid_points
    ]

    maximum_potential = max(
        value
        for _, value in potential_values
    )

    maximizer_points = tuple(
        point
        for point, value in potential_values
        if abs(
            value - maximum_potential
        ) <= tolerances.objective
    )

    return ExactRegionalPotential(
        region_id=region.id,
        maximum_potential=maximum_potential,
        maximizer_points=maximizer_points,
        lipschitz_constant=lipschitz_constant,
    )

In [49]:
@dataclass(frozen=True)
class CandidateSet:
    region_id: int
    sampling_candidates: tuple[Candidate, ...]
    split_candidates: tuple[SplitCandidate, ...]

    @property
    def is_empty(self) -> bool:
        return not (
            self.sampling_candidates
            or self.split_candidates
        )

    @property
    def count(self) -> int:
        return (
            len(self.sampling_candidates)
            + len(self.split_candidates)
        )


def generate_exact_candidates(
    view: RegionEvaluationView,
    config: ARRGOConfig,
) -> CandidateSet:
    region = view.region

    sampling_candidates = generate_sampling_candidates(
        view=view,
        tolerances=config.tolerances,
    )

    split_candidates = generate_split_candidates(
        region=region,
        contraction_factor=config.contraction_factor,
        tolerances=config.tolerances,
    )

    valid_split_candidates = tuple(
        candidate
        for candidate in split_candidates
        if satisfies_contraction_guarantee(
            region=region,
            split_location=candidate.split_location,
            contraction_factor=candidate.contraction_factor,
            tolerance=config.tolerances.point,
        )
    )

    return CandidateSet(
        region_id=region.id,
        sampling_candidates=sampling_candidates,
        split_candidates=valid_split_candidates,
    )


def generate_candidates_for_active_region(
    region: Region,
    state: ARRGOGlobalState,
) -> CandidateSet:
    if region.state != RegionState.ACTIVE:
        return CandidateSet(
            region_id=region.id,
            sampling_candidates=tuple(),
            split_candidates=tuple(),
        )

    view = build_shared_region_view(
        region=region,
        history=state.history,
        tolerance=state.config.tolerances.point,
    )

    return generate_exact_candidates(
        view=view,
        config=state.config,
    )

In [50]:
def build_split_profiles(
    view: RegionEvaluationView,
    split_candidates: tuple[SplitCandidate, ...],
    lipschitz_constant: float | None,
    tolerances: NumericalTolerances,
) -> tuple[StructuralDifferenceProfile, ...]:
    profiles: list[StructuralDifferenceProfile] = []

    for candidate in split_candidates:
        structural_value = evaluate_structural_split(
            candidate=candidate,
        )

        structural_difference = evaluate_structural_difference(
            candidate=candidate,
            view=view,
            tolerances=tolerances,
        )

        directional_difference = evaluate_directional_difference(
            candidate=candidate,
            view=view,
            tolerances=tolerances,
        )

        slope_variation_difference = (
            evaluate_slope_variation_difference(
                candidate=candidate,
                view=view,
                tolerances=tolerances,
            )
        )

        sampling_density_difference = (
            evaluate_sampling_density_difference(
                candidate=candidate,
                view=view,
                tolerances=tolerances,
            )
        )

        uncertainty_difference = (
            evaluate_uncertainty_difference(
                candidate=candidate,
                view=view,
                lipschitz_constant=lipschitz_constant,
                tolerances=tolerances,
            )
        )

        optimization_potential_difference = (
            evaluate_optimization_potential_difference(
                candidate=candidate,
                view=view,
                lipschitz_constant=lipschitz_constant,
                tolerances=tolerances,
            )
        )

        profiles.append(
            StructuralDifferenceProfile(
                structural_value=structural_value,
                structural_difference=structural_difference,
                directional_difference=directional_difference,
                slope_variation_difference=slope_variation_difference,
                sampling_density_difference=sampling_density_difference,
                uncertainty_difference=uncertainty_difference,
                optimization_potential_difference=(
                    optimization_potential_difference
                ),
            )
        )

    return tuple(profiles)


def select_exact_refinement_action(
    view: RegionEvaluationView,
    analysis: RegionAnalysis,
    stability: DecisionStability,
    candidate_set: CandidateSet,
    tolerances: NumericalTolerances,
) -> tuple[
    RefinementAction,
    SamplingCandidateEvaluation | None,
    StructuralDifferenceProfile | None,
]:
    if stability.is_stable:
        return (
            RefinementAction.STABLE,
            None,
            None,
        )

    sampling_evaluations = tuple(
        evaluate_sampling_candidate(
            candidate=candidate,
            view=view,
            tolerances=tolerances,
        )
        for candidate in candidate_set.sampling_candidates
    )

    sampling_candidate = select_sampling_candidate(
        evaluations=sampling_evaluations,
        tolerances=tolerances,
    )

    split_profiles = build_split_profiles(
        view=view,
        split_candidates=candidate_set.split_candidates,
        lipschitz_constant=None,
        tolerances=tolerances,
    )

    split_candidate = select_non_dominated_split(
        profiles=split_profiles,
        tolerances=tolerances,
    )

    if sampling_candidate is None and split_candidate is None:
        return (
            RefinementAction.STABLE,
            None,
            None,
        )

    if sampling_candidate is not None and split_candidate is None:
        return (
            RefinementAction.SAMPLE,
            sampling_candidate,
            None,
        )

    if sampling_candidate is None and split_candidate is not None:
        return (
            RefinementAction.SPLIT,
            None,
            split_candidate,
        )

    assert sampling_candidate is not None
    assert split_candidate is not None

    if (
        analysis.coverage_resolution
        > analysis.width * 0.5
        and sampling_candidate.improves_coverage
    ):
        return (
            RefinementAction.SAMPLE,
            sampling_candidate,
            split_candidate,
        )

    return (
        RefinementAction.SPLIT,
        sampling_candidate,
        split_candidate,
    )

In [51]:
@dataclass(frozen=True)
class IterationResult:
    iteration: int
    selected_region_id: int | None
    action: RefinementAction
    evaluation: Evaluation | None
    created_region_ids: tuple[int, ...]
    termination_status: TerminationStatus


def build_region_resolution_state(
    view: RegionEvaluationView,
    config: ARRGOConfig,
) -> UnresolvedInformationState:
    coverage = evaluate_coverage_resolution_objective(
        view=view,
        tolerances=config.tolerances,
    )

    behavior = evaluate_behavior_resolution_objective(
        view=view,
        tolerances=config.tolerances,
    )

    uncertainty = evaluate_uncertainty_resolution_objective(
        view=view,
        lipschitz_constant=config.lipschitz_constant,
        tolerances=config.tolerances,
    )

    potential = (
        evaluate_optimization_potential_resolution_objective(
            view=view,
            lipschitz_constant=config.lipschitz_constant,
        )
    )

    return build_unresolved_information_state(
        coverage=coverage,
        behavior=behavior,
        uncertainty=uncertainty,
        potential=potential,
    )


def execute_arrgo_iteration(
    state: ARRGOGlobalState,
    evaluator: ObjectiveEvaluator,
    criteria: ResolutionCriteria,
    iteration: int,
) -> IterationResult:
    if iteration < 0:
        raise ValueError(
            "Iteration must be non-negative."
        )

    active_regions = state.active_regions

    if not active_regions:
        status = TerminationStatus(
            terminated=True,
            reason=TerminationReason.NO_VALID_ACTION,
            message="No active regions remain.",
        )

        return IterationResult(
            iteration=iteration,
            selected_region_id=None,
            action=RefinementAction.STABLE,
            evaluation=None,
            created_region_ids=tuple(),
            termination_status=status,
        )

    actions: dict[int, RefinementAction] = {}
    analyses: dict[int, RegionAnalysis] = {}
    resolution_states: dict[
        int,
        UnresolvedInformationState,
    ] = {}

    candidate_sets: dict[
        int,
        CandidateSet,
    ] = {}

    selected_sampling: dict[
        int,
        SamplingCandidateEvaluation | None,
    ] = {}

    selected_splits: dict[
        int,
        StructuralDifferenceProfile | None,
    ] = {}

    for region in active_regions:
        view = build_shared_region_view(
            region=region,
            history=state.history,
            tolerance=state.config.tolerances.point,
        )

        analysis = analyze_region(
            view=view,
            tolerances=state.config.tolerances,
        )

        resolution_state = build_region_resolution_state(
            view=view,
            config=state.config,
        )

        stability = evaluate_decision_stability(
            state=resolution_state,
            criteria=criteria,
        )

        candidates = generate_exact_candidates(
            view=view,
            config=state.config,
        )

        action, sampling_candidate, split_candidate = (
            select_exact_refinement_action(
                view=view,
                analysis=analysis,
                stability=stability,
                candidate_set=candidates,
                tolerances=state.config.tolerances,
            )
        )

        actions[region.id] = action
        analyses[region.id] = analysis
        resolution_states[region.id] = resolution_state
        candidate_sets[region.id] = candidates
        selected_sampling[region.id] = sampling_candidate
        selected_splits[region.id] = split_candidate

    priorities = tuple(
        calculate_global_region_priority(
            state=resolution_states[region.id],
        )
        for region in active_regions
        if actions[region.id] != RefinementAction.STABLE
    )

    selected_priority = select_highest_priority_region(
        priorities=priorities,
    )

    global_gap = None

    termination_status = evaluate_termination(
        state=state,
        global_gap=global_gap,
        actions=actions,
    )

    if termination_status.terminated:
        return IterationResult(
            iteration=iteration,
            selected_region_id=None,
            action=RefinementAction.STABLE,
            evaluation=None,
            created_region_ids=tuple(),
            termination_status=termination_status,
        )

    if selected_priority is None:
        status = TerminationStatus(
            terminated=True,
            reason=TerminationReason.NO_VALID_ACTION,
            message="No region with a valid refinement action was found.",
        )

        return IterationResult(
            iteration=iteration,
            selected_region_id=None,
            action=RefinementAction.STABLE,
            evaluation=None,
            created_region_ids=tuple(),
            termination_status=status,
        )

    selected_region = state.hierarchy.get(
        selected_priority.region_id
    )

    action = actions[selected_region.id]

    if action == RefinementAction.SAMPLE:
        sampling_candidate = selected_sampling[
            selected_region.id
        ]

        if sampling_candidate is None:
            raise RuntimeError(
                "A sampling action requires a valid sampling candidate."
            )

        if not evaluator.can_evaluate(
            sampling_candidate.candidate.location
        ):
            raise RuntimeError(
                "The selected sampling candidate cannot be evaluated."
            )

        evaluation = evaluator.evaluate(
            sampling_candidate.candidate.location
        )

        update_global_state_after_evaluation(
            state=state,
            evaluation=evaluation,
        )

        return IterationResult(
            iteration=iteration,
            selected_region_id=selected_region.id,
            action=RefinementAction.SAMPLE,
            evaluation=evaluation,
            created_region_ids=tuple(),
            termination_status=TerminationStatus(
                terminated=False,
                reason=None,
                message="Sampling action completed.",
            ),
        )

    if action == RefinementAction.SPLIT:
        split_candidate = selected_splits[
            selected_region.id
        ]

        if split_candidate is None:
            raise RuntimeError(
                "A split action requires a valid split candidate."
            )

        validate_split_contraction(
            region=selected_region,
            split_candidate=split_candidate.candidate,
            tolerance=state.config.tolerances.point,
        )

        left_child, right_child = (
            create_child_regions_from_split(
                hierarchy=state.hierarchy,
                split_candidate=split_candidate.candidate,
            )
        )

        activate_children_after_split(
            hierarchy=state.hierarchy,
            parent_id=selected_region.id,
        )

        return IterationResult(
            iteration=iteration,
            selected_region_id=selected_region.id,
            action=RefinementAction.SPLIT,
            evaluation=None,
            created_region_ids=(
                left_child.id,
                right_child.id,
            ),
            termination_status=TerminationStatus(
                terminated=False,
                reason=None,
                message="Structural split completed.",
            ),
        )

    raise RuntimeError(
        f"Unsupported action selected: {action}"
    )

In [52]:
@dataclass
class ARRGOExecutionCycle:
    state: ARRGOGlobalState
    evaluator: ObjectiveEvaluator
    criteria: ResolutionCriteria
    iteration: int = 0

    def __post_init__(self) -> None:
        if self.iteration < 0:
            raise ValueError(
                "Iteration must be non-negative."
            )

    @property
    def is_budget_exhausted(self) -> bool:
        return self.evaluator.budget_exhausted

    @property
    def current_iteration(self) -> int:
        return self.iteration

    def run_iteration(self) -> IterationResult:
        result = execute_arrgo_iteration(
            state=self.state,
            evaluator=self.evaluator,
            criteria=self.criteria,
            iteration=self.iteration,
        )

        self.iteration += 1

        return result

    def run_until_termination(
        self,
        max_iterations: int | None = None,
    ) -> tuple[IterationResult, ...]:
        if max_iterations is not None and max_iterations < 0:
            raise ValueError(
                "Maximum iterations must be non-negative."
            )

        results: list[IterationResult] = []

        while True:
            if (
                max_iterations is not None
                and len(results) >= max_iterations
            ):
                break

            result = self.run_iteration()

            results.append(result)

            if result.termination_status.terminated:
                break

        return tuple(results)

In [53]:
def calculate_global_best_value(
    state: ARRGOGlobalState,
) -> float | None:
    return state.best_value


def calculate_global_potential(
    state: ARRGOGlobalState,
) -> float | None:
    if state.config.mode != ARRGOExecutionMode.CERTIFIED:
        return None

    potentials: list[float] = []

    for region in state.hierarchy.regions:
        view = build_shared_region_view(
            region=region,
            history=state.history,
            tolerance=state.config.tolerances.point,
        )

        regional_potential = calculate_exact_regional_potential(
            view=view,
            lipschitz_constant=state.config.lipschitz_constant,
            tolerances=state.config.tolerances,
        )

        if regional_potential is not None:
            potentials.append(
                regional_potential.maximum_potential
            )

    if not potentials:
        return None

    return max(potentials)


def calculate_global_gap(
    state: ARRGOGlobalState,
) -> float | None:
    if state.config.mode != ARRGOExecutionMode.CERTIFIED:
        return None

    best_value = calculate_global_best_value(
        state=state,
    )

    global_potential = calculate_global_potential(
        state=state,
    )

    if (
        best_value is None
        or global_potential is None
    ):
        return None

    return max(
        0.0,
        global_potential - best_value,
    )


def run_global_selection_and_refinement_loop(
    state: ARRGOGlobalState,
    evaluator: ObjectiveEvaluator,
    criteria: ResolutionCriteria,
    max_iterations: int | None = None,
) -> tuple[IterationResult, ...]:
    if max_iterations is not None and max_iterations < 0:
        raise ValueError(
            "Maximum iterations must be non-negative."
        )

    results: list[IterationResult] = []
    iteration = 0

    while True:
        if (
            max_iterations is not None
            and iteration >= max_iterations
        ):
            break

        active_regions = state.active_regions

        if not active_regions:
            results.append(
                IterationResult(
                    iteration=iteration,
                    selected_region_id=None,
                    action=RefinementAction.STABLE,
                    evaluation=None,
                    created_region_ids=tuple(),
                    termination_status=TerminationStatus(
                        terminated=True,
                        reason=TerminationReason.NO_VALID_ACTION,
                        message="No active regions remain.",
                    ),
                )
            )

            break

        actions: dict[int, RefinementAction] = {}
        resolution_states: dict[
            int,
            UnresolvedInformationState,
        ] = {}

        sampling_candidates: dict[
            int,
            SamplingCandidateEvaluation | None,
        ] = {}

        split_candidates: dict[
            int,
            StructuralDifferenceProfile | None,
        ] = {}

        for region in active_regions:
            view = build_shared_region_view(
                region=region,
                history=state.history,
                tolerance=state.config.tolerances.point,
            )

            analysis = analyze_region(
                view=view,
                tolerances=state.config.tolerances,
            )

            resolution_state = build_region_resolution_state(
                view=view,
                config=state.config,
            )

            stability = evaluate_decision_stability(
                state=resolution_state,
                criteria=criteria,
            )

            candidate_set = generate_exact_candidates(
                view=view,
                config=state.config,
            )

            (
                action,
                sampling_candidate,
                split_candidate,
            ) = select_exact_refinement_action(
                view=view,
                analysis=analysis,
                stability=stability,
                candidate_set=candidate_set,
                tolerances=state.config.tolerances,
            )

            actions[region.id] = action
            resolution_states[region.id] = resolution_state
            sampling_candidates[region.id] = sampling_candidate
            split_candidates[region.id] = split_candidate

        global_gap = calculate_global_gap(
            state=state,
        )

        termination_status = evaluate_termination(
            state=state,
            global_gap=global_gap,
            actions=actions,
        )

        if termination_status.terminated:
            results.append(
                IterationResult(
                    iteration=iteration,
                    selected_region_id=None,
                    action=RefinementAction.STABLE,
                    evaluation=None,
                    created_region_ids=tuple(),
                    termination_status=termination_status,
                )
            )

            break

        priorities = tuple(
            calculate_global_region_priority(
                state=resolution_states[region.id],
            )
            for region in active_regions
            if actions[region.id] != RefinementAction.STABLE
        )

        selected_priority = select_highest_priority_region(
            priorities=priorities,
        )

        if selected_priority is None:
            results.append(
                IterationResult(
                    iteration=iteration,
                    selected_region_id=None,
                    action=RefinementAction.STABLE,
                    evaluation=None,
                    created_region_ids=tuple(),
                    termination_status=TerminationStatus(
                        terminated=True,
                        reason=TerminationReason.NO_VALID_ACTION,
                        message="No valid global refinement action remains.",
                    ),
                )
            )

            break

        selected_region = state.hierarchy.get(
            selected_priority.region_id
        )

        action = actions[selected_region.id]

        if action == RefinementAction.SAMPLE:
            selected_sampling = sampling_candidates[
                selected_region.id
            ]

            if selected_sampling is None:
                raise RuntimeError(
                    "Selected sampling action has no candidate."
                )

            location = selected_sampling.candidate.location

            if not evaluator.can_evaluate(location):
                raise RuntimeError(
                    "Selected sampling location cannot be evaluated."
                )

            evaluation = evaluator.evaluate(location)

            update_global_state_after_evaluation(
                state=state,
                evaluation=evaluation,
            )

            result = IterationResult(
                iteration=iteration,
                selected_region_id=selected_region.id,
                action=RefinementAction.SAMPLE,
                evaluation=evaluation,
                created_region_ids=tuple(),
                termination_status=TerminationStatus(
                    terminated=False,
                    reason=None,
                    message="Global sampling refinement completed.",
                ),
            )

        elif action == RefinementAction.SPLIT:
            selected_split = split_candidates[
                selected_region.id
            ]

            if selected_split is None:
                raise RuntimeError(
                    "Selected split action has no candidate."
                )

            split = selected_split.candidate

            validate_split_contraction(
                region=selected_region,
                split_candidate=split,
                tolerance=state.config.tolerances.point,
            )

            left_child, right_child = (
                create_child_regions_from_split(
                    hierarchy=state.hierarchy,
                    split_candidate=split,
                )
            )

            activate_children_after_split(
                hierarchy=state.hierarchy,
                parent_id=selected_region.id,
            )

            result = IterationResult(
                iteration=iteration,
                selected_region_id=selected_region.id,
                action=RefinementAction.SPLIT,
                evaluation=None,
                created_region_ids=(
                    left_child.id,
                    right_child.id,
                ),
                termination_status=TerminationStatus(
                    terminated=False,
                    reason=None,
                    message="Global structural refinement completed.",
                ),
            )

        else:
            raise RuntimeError(
                f"Unsupported global action: {action}"
            )

        results.append(result)
        iteration += 1

    return tuple(results)

In [54]:
@dataclass(frozen=True)
class EvaluationBudget:
    maximum_evaluations: int
    used_evaluations: int

    def __post_init__(self) -> None:
        if self.maximum_evaluations < 0:
            raise ValueError(
                "Maximum evaluations must be non-negative."
            )

        if self.used_evaluations < 0:
            raise ValueError(
                "Used evaluations must be non-negative."
            )

        if self.used_evaluations > self.maximum_evaluations:
            raise ValueError(
                "Used evaluations cannot exceed the maximum budget."
            )

    @property
    def remaining_evaluations(self) -> int:
        return (
            self.maximum_evaluations
            - self.used_evaluations
        )

    @property
    def is_exhausted(self) -> bool:
        return self.remaining_evaluations == 0

    def can_evaluate(self) -> bool:
        return not self.is_exhausted


def build_evaluation_budget(
    state: ARRGOGlobalState,
) -> EvaluationBudget:
    return EvaluationBudget(
        maximum_evaluations=state.config.max_evaluations,
        used_evaluations=state.evaluation_count,
    )


def can_execute_sampling_action(
    state: ARRGOGlobalState,
    candidate: SamplingCandidateEvaluation | None,
) -> bool:
    if candidate is None:
        return False

    budget = build_evaluation_budget(
        state=state,
    )

    if not budget.can_evaluate():
        return False

    return not state.history.contains_point(
        candidate.candidate.location
    )


def record_budget_usage(
    state: ARRGOGlobalState,
) -> EvaluationBudget:
    budget = build_evaluation_budget(
        state=state,
    )

    if budget.used_evaluations > budget.maximum_evaluations:
        raise RuntimeError(
            "Evaluation budget has been exceeded."
        )

    return budget

In [55]:
def is_numerically_zero(
    value: float,
    tolerance: float,
) -> bool:
    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    return abs(value) <= tolerance


def are_points_equal(
    first: float,
    second: float,
    tolerance: float,
) -> bool:
    if tolerance < 0.0:
        raise ValueError(
            "Tolerance must be non-negative."
        )

    return abs(first - second) <= tolerance


def validate_region_numerical_geometry(
    region: Region,
    tolerances: NumericalTolerances,
) -> None:
    if not __import__("math").isfinite(
        region.lower_bound
    ):
        raise ValueError(
            "Region lower bound must be finite."
        )

    if not __import__("math").isfinite(
        region.upper_bound
    ):
        raise ValueError(
            "Region upper bound must be finite."
        )

    if region.width <= tolerances.boundary:
        raise ValueError(
            "Region width is too small for reliable numerical refinement."
        )


def validate_split_numerical_geometry(
    region: Region,
    split_location: float,
    tolerances: NumericalTolerances,
) -> None:
    if not __import__("math").isfinite(
        split_location
    ):
        raise ValueError(
            "Split location must be finite."
        )

    if not region.contains(
        split_location,
        tolerances.boundary,
    ):
        raise ValueError(
            "Split location lies outside the region."
        )

    left_width = (
        split_location
        - region.lower_bound
    )

    right_width = (
        region.upper_bound
        - split_location
    )

    if left_width <= tolerances.boundary:
        raise ValueError(
            "Split produces a numerically degenerate left child."
        )

    if right_width <= tolerances.boundary:
        raise ValueError(
            "Split produces a numerically degenerate right child."
        )


def validate_evaluation_numerics(
    evaluation: Evaluation,
) -> None:
    if not __import__("math").isfinite(
        evaluation.x
    ):
        raise ValueError(
            "Evaluation point must be finite."
        )

    if not __import__("math").isfinite(
        evaluation.value
    ):
        raise ValueError(
            "Evaluation value must be finite."
        )


def validate_candidate_numerics(
    candidate: Candidate,
    region: Region,
    tolerances: NumericalTolerances,
) -> None:
    if not __import__("math").isfinite(
        candidate.location
    ):
        raise ValueError(
            "Candidate location must be finite."
        )

    if candidate.region_id != region.id:
        raise ValueError(
            "Candidate does not belong to the given region."
        )

    if not region.contains(
        candidate.location,
        tolerances.boundary,
    ):
        raise ValueError(
            "Candidate location lies outside the region."
        )


def validate_global_state_numerics(
    state: ARRGOGlobalState,
) -> None:
    if state.evaluation_count < 0:
        raise RuntimeError(
            "Evaluation count cannot be negative."
        )

    if (
        state.evaluation_count
        > state.config.max_evaluations
    ):
        raise RuntimeError(
            "Evaluation budget has been exceeded."
        )

    if (
        state.incumbent.evaluation_count
        != state.evaluation_count
    ):
        raise RuntimeError(
            "Global incumbent count is inconsistent "
            "with evaluation history."
        )

    for region in state.hierarchy.regions:
        validate_region_numerical_geometry(
            region=region,
            tolerances=state.config.tolerances,
        )

In [56]:
@dataclass(frozen=True)
class ARRGOResult:
    best_point: float | None
    best_value: float | None
    evaluation_count: int
    iteration_count: int
    termination_reason: TerminationReason | None
    history: tuple[Evaluation, ...]
    regions: tuple[Region, ...]
    iterations: tuple[IterationResult, ...]

    @property
    def has_solution(self) -> bool:
        return (
            self.best_point is not None
            and self.best_value is not None
        )


class ARRGO:
    def __init__(
        self,
        objective: ObjectiveFunction,
        config: ARRGOConfig,
        criteria: ResolutionCriteria,
    ) -> None:
        self._objective = objective
        self._config = config
        self._criteria = criteria

        self._state = create_initial_global_state(
            config=config,
        )

        self._evaluator = ObjectiveEvaluator(
            objective=objective,
            history=self._state.history,
            max_evaluations=config.max_evaluations,
        )

        self._iterations: list[IterationResult] = []

    @property
    def state(self) -> ARRGOGlobalState:
        return self._state

    @property
    def evaluator(self) -> ObjectiveEvaluator:
        return self._evaluator

    @property
    def iterations(self) -> tuple[IterationResult, ...]:
        return tuple(self._iterations)

    def initialize(
        self,
        initial_points: tuple[float, ...],
    ) -> None:
        if self._state.evaluation_count > 0:
            raise RuntimeError(
                "ARRGO has already been initialized."
            )

        if not initial_points:
            raise ValueError(
                "At least one initial point is required."
            )

        for point in initial_points:
            if not self._evaluator.can_evaluate(point):
                raise ValueError(
                    f"Initial point x={point} cannot be evaluated."
                )

            evaluation = self._evaluator.evaluate(point)

            update_global_state_after_evaluation(
                state=self._state,
                evaluation=evaluation,
            )

    def run(
        self,
        initial_points: tuple[float, ...],
        max_iterations: int | None = None,
    ) -> ARRGOResult:
        if max_iterations is not None and max_iterations < 0:
            raise ValueError(
                "Maximum iterations must be non-negative."
            )

        self.initialize(
            initial_points=initial_points,
        )

        results = run_global_selection_and_refinement_loop(
            state=self._state,
            evaluator=self._evaluator,
            criteria=self._criteria,
            max_iterations=max_iterations,
        )

        self._iterations.extend(results)

        if results:
            final_status = results[-1].termination_status
            termination_reason = final_status.reason
        else:
            termination_reason = None

        validate_global_state_numerics(
            state=self._state,
        )

        return ARRGOResult(
            best_point=self._state.best_point,
            best_value=self._state.best_value,
            evaluation_count=self._state.evaluation_count,
            iteration_count=len(self._iterations),
            termination_reason=termination_reason,
            history=self._state.history.evaluations,
            regions=self._state.hierarchy.regions,
            iterations=self.iterations,
        )

In [57]:
objective = CallableObjective(
    lambda x: -(x - 2.0) ** 2 + 5.0
)

config = ARRGOConfig(
    lower_bound=-5.0,
    upper_bound=5.0,
    max_evaluations=50,
)

criteria = ResolutionCriteria(
    coverage_tolerance=1e-3,
    behavior_tolerance=1e-3,
    uncertainty_tolerance=1e-3,
    potential_tolerance=1e-3,
)

optimizer = ARRGO(
    objective=objective,
    config=config,
    criteria=criteria,
)

result = optimizer.run(
    initial_points=(-5.0, 0.0, 5.0),
)

print("Best point:", result.best_point)
print("Best value:", result.best_value)
print("Evaluations:", result.evaluation_count)
print("Termination:", result.termination_reason)

Best point: 1.875
Best value: 4.984375
Evaluations: 50
Termination: TerminationReason.BUDGET_EXHAUSTED
